# Notebook 3a - How a convolutional neural network works

**UKACM Autumn School: AI for Computational Mechanics**

### Where Notebook 2 left off

Notebook 2 gave a neural network the raw 64x64 microstructure image. To do that, the image was
flattened into a list of 4096 numbers. The network then treated pixel 100 and pixel 101 as two
unrelated inputs, even though they sit next to each other. The test at the end showed this:
moving every pixel to a random position did not change the score.

This notebook builds the layer that does know which pixels are neighbours: the **convolution**.
A network made of these layers is a **convolutional neural network**, or CNN.

### You already know the main idea

If you have written a finite-difference scheme, you have already used a convolution. A stencil
such as

$$\frac{du}{dx}\bigg|_i \approx \frac{u_{i+1} - u_{i-1}}{2h}$$

is a small set of weights, $(-\tfrac{1}{2h},\ 0,\ \tfrac{1}{2h})$, applied at every node of the grid.
A CNN does the same thing, except that its weights come from training on data rather than from a
Taylor series.

### How this notebook is organised

Each idea is shown on a small example you could check with a calculator, and most have an
animation. No large model is trained here. Notebook 3b applies the same ideas to the
microstructures.

| Part | Question | Main picture |
|---|---|---|
| 1 | What does a convolution compute? | a stencil sliding along a line, then across an image |
| 2 | How is a convolution layer related to the neurons of Notebook 2? | a dense layer turning into a convolution |
| 3 | What else goes into a CNN? | padding, stride, channels, pooling, how far a unit can see |
| 4 | How does a convolution learn its weights? | the backward pass, number by number |
| 5 | Why does any of this help? | two small experiments |

**How to use it.** Run the cells in order. Animation cells show a player: press play, or step
frame by frame with the arrow buttons. You do not need to read the code inside animation cells. The
text above each one says what to look for.

In [ ]:
# --- Setup -------------------------------------------------------------------
# Run this first.
import os, io, time, zipfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyArrowPatch
from IPython.display import HTML, display
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Text

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)          # matches a free Colab CPU runtime

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 60,
})
C_DATA, C_FIT, C_ALT, C_BAD = "#3B6EA5", "#C25E00", "#4C9A5E", "#A8323E"
ANIM_DPI = 80                     # animations are rendered a little smaller to keep the notebook light

print("numpy", np.__version__, "| torch", torch.__version__)


In [ ]:
# --- Drawing helpers ---------------------------------------------------------
# Small functions used by the figures and animations. You do not need to read them.

def bare(ax):
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

def number_grid(ax, shape, cmap="coolwarm", vmax=1.0, title="", fontsize=9, gridlines=True):
    # An empty grid of cells that can later be filled with values and printed numbers.
    cm = plt.get_cmap(cmap).copy()
    cm.set_bad("0.94")                                   # cells with no value yet
    im = ax.imshow(np.full(shape, np.nan), cmap=cm, vmin=-vmax, vmax=vmax,
                   interpolation="nearest")
    if gridlines:
        for g in range(shape[0] + 1):
            ax.axhline(g - 0.5, color="0.65", lw=0.6)
        for g in range(shape[1] + 1):
            ax.axvline(g - 0.5, color="0.65", lw=0.6)
    bare(ax)
    ax.set_title(title, fontsize=10)
    texts = [[ax.text(c, r, "", ha="center", va="center", fontsize=fontsize)
              for c in range(shape[1])] for r in range(shape[0])]
    return {"im": im, "tx": texts, "ax": ax}

def fill_grid(g, arr, fmt="{:g}"):
    a = np.asarray(arr, dtype=float)
    g["im"].set_data(a)
    for r in range(a.shape[0]):
        for c in range(a.shape[1]):
            v = a[r, c]
            t = g["tx"][r][c]
            if np.isnan(v):
                t.set_text("")
                continue
            t.set_text("0" if v == 0 else fmt.format(v))
            shade = g["im"].norm(v)
            t.set_color("w" if (shade < 0.15 or shade > 0.85) else "k")

def outline(ax, row, col, h=1, w=1, color=C_FIT, lw=3, store=None):
    p = Rectangle((col - 0.5, row - 0.5), w, h, fill=False, ec=color, lw=lw, zorder=6)
    ax.add_patch(p)
    if store is not None:
        store.append(p)
    return p

def play(anim, fig):
    # Close the static figure and show the animation player instead.
    plt.close(fig)
    return HTML(anim.to_jshtml())

def conv2d_valid(img, k):
    # The convolution written as two loops, exactly as in the equation. No padding.
    kh, kw = k.shape
    oh, ow = img.shape[0] - kh + 1, img.shape[1] - kw + 1
    out = np.empty((oh, ow), dtype=np.float32)
    for i in range(oh):
        for j in range(ow):
            out[i, j] = float((img[i:i+kh, j:j+kw] * k).sum())
    return out

print("helpers ready")


### Loading the data

Only one file is needed here: `microstructures_64.npz`, the 1485 binary microstructure images at
64x64 pixels. In every image, 1 is fibre and 0 is matrix. Notebook 3b uses the labels as well.

The following cell downloads the shared CNN dataset automatically.

In [ ]:
# --- Load data ---------------------------------------------------------------
DATA_URL = "https://raw.githubusercontent.com/CEMS-Lab/autumn-school/main/datasets/machine_learning/NB3_data.zip"  # shared CNN dataset
DATA_DIR = "."
FILES = ["microstructures_64.npz"]

def _have():
    return all(os.path.exists(os.path.join(DATA_DIR, f)) for f in FILES)

if not _have() and DATA_URL:
    print("Downloading ...")
    with urllib.request.urlopen(DATA_URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)

if not _have():
    try:
        from google.colab import files
        print("Select:", ", ".join(FILES))
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put " + ", ".join(FILES) + " beside the notebook, or set DATA_URL above.")

_img = np.load(os.path.join(DATA_DIR, "microstructures_64.npz"), allow_pickle=True)
X_IMG = _img["images"].astype(np.float32)
IMG_DEMO = 900                     # a cell with about 40% fibre, used in several figures below
print(f"{X_IMG.shape[0]} images of {X_IMG.shape[1]}x{X_IMG.shape[2]} pixels, values {np.unique(X_IMG)}")

fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax, i in zip(axes, [0, 300, 600, IMG_DEMO, 1200]):
    ax.imshow(X_IMG[i], cmap="gray", interpolation="nearest")
    ax.set_title(f"image {i}, fibre fraction {X_IMG[i].mean():.2f}", fontsize=8)
    bare(ax)
plt.tight_layout(); plt.show()


---

# Part 1 - A convolution is a stencil

### 1.1 One dimension first

Take one row of pixels from a microstructure. Along that row the value jumps between 0 (matrix) and
1 (fibre). Call the values $u_0, u_1, u_2, \dots$. They are values on a grid with spacing
$h = 1$ pixel, like the nodal values in a 1D finite-difference problem.

Now slide a set of three weights $w_0, w_1, w_2$ along the row. At each position $i$, multiply
each weight by the value underneath it and add up the three products:

$$\boxed{\;o_i \;=\; w_0\,u_i \;+\; w_1\,u_{i+1} \;+\; w_2\,u_{i+2} \;=\; \sum_{m=0}^{2} w_m\, u_{i+m}\;}$$

This operation is a **1D convolution**. The three weights are the **kernel**. The list of results $o_i$ is
the **output**, and in a CNN it is called a **feature map**.

Two kernels to try:

| Kernel $(w_0, w_1, w_2)$ | What it computes | Finite-difference name |
|---|---|---|
| $(-\tfrac12,\ 0,\ \tfrac12)$ | slope of $u$ at the middle node | central difference for $du/dx$ |
| $(\tfrac13,\ \tfrac13,\ \tfrac13)$ | average of three neighbours | local fibre fraction |

Each window covers nodes $i$, $i+1$, $i+2$, so $o_i$ belongs to the middle node $i+1$. The plots
below draw it there.

In [ ]:
# --- One row of a real microstructure -----------------------------------------
IMG_1D, ROW_1D, SEG = 273, 54, 40            # image, row, and how many pixels of the row to use
U = X_IMG[IMG_1D, ROW_1D, :SEG]

W_DIFF = np.array([-0.5, 0.0, 0.5])          # central difference
W_AVG  = np.array([1, 1, 1]) / 3.0           # three-point average

def conv1d_valid(u, w):
    k = len(w)
    return np.array([np.sum(w * u[i:i+k]) for i in range(len(u) - k + 1)])

O_DIFF = conv1d_valid(U, W_DIFF)
O_AVG  = conv1d_valid(U, W_AVG)

print("u      :", U.astype(int))
print("o diff :", O_DIFF)
print(f"{len(U)} input values and a kernel of 3 give {len(O_DIFF)} outputs")

fig, (a0, a1) = plt.subplots(2, 1, figsize=(11, 3.6), gridspec_kw={"height_ratios": [1, 1.3]})
a0.imshow(X_IMG[IMG_1D, ROW_1D-8:ROW_1D+9, :SEG], cmap="gray", interpolation="nearest")
a0.axhline(8, color=C_FIT, lw=2)
a0.set_title(f"image {IMG_1D}, rows {ROW_1D-8} to {ROW_1D+8}; the orange line is row {ROW_1D}", fontsize=9)
bare(a0)
a1.step(np.arange(SEG), U, where="mid", color="k")
a1.set_xlim(-0.5, SEG - 0.5); a1.set_ylim(-0.2, 1.2)
a1.set_xlabel("pixel $i$"); a1.set_ylabel("$u_i$")
a1.set_yticks([0, 1]); a1.set_yticklabels(["0 matrix", "1 fibre"])
plt.tight_layout(); plt.show()


### Animation 1: the kernel sliding along the row

**Top:** the image around the row, with the current window in orange. **Middle:** the signal
$u_i$, with the three weights of the difference kernel written under the three values they
multiply. **Bottom:** the two outputs, filled in one position at a time. Red and blue bars are the
difference kernel, the green line is the average kernel.

Look at where the red and blue bars appear, and at what the green line does inside a fibre.

In [ ]:
# --- Animation 1: a 1D convolution --------------------------------------------
fig = plt.figure(figsize=(14, 7), dpi=ANIM_DPI)
gsp = fig.add_gridspec(3, 1, height_ratios=[1.0, 1.3, 2.0], hspace=0.45)
axT, axM, axB = [fig.add_subplot(gsp[k]) for k in range(3)]

axT.imshow(X_IMG[IMG_1D, ROW_1D-4:ROW_1D+5, :SEG], cmap="gray", interpolation="nearest", aspect="auto")
axT.set_xlim(-0.5, SEG - 0.5)
axT.axhline(4 - 0.5, color=C_FIT, lw=0.8); axT.axhline(4 + 0.5, color=C_FIT, lw=0.8)
bare(axT)
winT = Rectangle((-0.5, 3.5), 3, 1, fill=False, ec=C_FIT, lw=3, zorder=5)
axT.add_patch(winT)

axM.bar(np.arange(SEG), U, width=0.8, color="0.55")
axM.set_xlim(-0.5, SEG - 0.5); axM.set_ylim(-0.9, 1.35)
axM.set_yticks([0, 1]); axM.set_ylabel("$u_i$"); axM.set_xticks([])
winM = Rectangle((-0.5, -0.05), 3, 1.25, fc=C_FIT, alpha=0.18, ec=C_FIT, lw=2)
axM.add_patch(winM)
WLAB = ["-½", "0", "+½"]
wtxt = [axM.text(0, -0.5, WLAB[m], ha="center", fontsize=10, color=C_FIT, weight="bold") for m in range(3)]
axM.text(-0.3, -0.5, "weights:", ha="right", fontsize=9, color=C_FIT)
sum_txt = axM.text(SEG - 1, 1.22, "", ha="right", fontsize=11)

xs = np.arange(len(O_DIFF)) + 1                   # each output drawn at the middle node
bars = axB.bar(xs, np.zeros_like(O_DIFF), width=0.8, color="w")
avg_line, = axB.plot([], [], "-o", ms=3, color=C_ALT, lw=2, label="average kernel")
axB.axhline(0, color="k", lw=0.8)
axB.set_xlim(-0.5, SEG - 0.5); axB.set_ylim(-0.7, 1.15)
axB.set_xlabel("pixel $i$"); axB.set_ylabel("output $o$")
axB.bar([0], [0], color=C_BAD, label="difference kernel, positive")
axB.bar([0], [0], color=C_DATA, label="difference kernel, negative")
axB.legend(fontsize=8, loc="lower left", ncol=3)

def frame_1d(f):
    winT.set_x(f - 0.5); winM.set_x(f - 0.5)
    for m in range(3):
        wtxt[m].set_position((f + m, -0.5))
    for k, b in enumerate(bars):
        v = O_DIFF[k] if k <= f else 0.0
        b.set_height(v); b.set_color(C_BAD if v > 0 else C_DATA)
    avg_line.set_data(xs[:f+1], O_AVG[:f+1])
    sum_txt.set_text(f"o_{f} = ({-0.5:+g})({U[f]:g}) + (0)({U[f+1]:g}) + ({0.5:+g})({U[f+2]:g}) = {O_DIFF[f]:+.1f}")
    axT.set_title(f"window at i = {f}", fontsize=10)
    return []

anim = animation.FuncAnimation(fig, frame_1d, frames=len(O_DIFF), interval=350)
play(anim, fig)


**What the animation shows.** Inside a fibre and inside the matrix, the value is the same under
all three weights, so the difference kernel returns zero. It is non-zero only where the window
straddles a boundary: positive (red) where the row enters a fibre, negative (blue) where it leaves.
The output marks where the interfaces are.

The average kernel does something different. It returns 1 deep inside a fibre, 0 in the matrix, and
$\tfrac13$ or $\tfrac23$ at a boundary. It is a smoothed local fibre fraction.

The operation was the same both times and only the weights changed. A CNN works on this basis: it
adjusts its weights until the outputs carry the information needed for the prediction.

### Try it: your own 1D kernel

Move the three sliders and watch the output change. Things to try:

- $(0, 1, 0)$ copies the signal.
- $(1, -2, 1)$ is the second-difference stencil for $d^2u/dx^2$. How many bars does it give at each
  boundary, and why?
- $(-1, 0, 1)$ gives the same pattern as the difference kernel, twice as tall.
- Any three weights that add up to zero give zero inside a fibre and inside the matrix. Check it.

In [ ]:
# --- Interactive: a 1D kernel of your choice -----------------------------------
def kernel_1d(w0=-0.5, w1=0.0, w2=0.5):
    w = np.array([w0, w1, w2])
    o = conv1d_valid(U, w)
    fig, (a0, a1) = plt.subplots(2, 1, figsize=(10, 4.2), sharex=True)
    a0.bar(np.arange(SEG), U, color="0.55", width=0.8)
    a0.set_ylabel("$u_i$"); a0.set_yticks([0, 1])
    a0.set_title(f"kernel ({w0:+.2f}, {w1:+.2f}, {w2:+.2f}),  sum of weights = {w.sum():+.2f}", fontsize=10)
    a1.bar(np.arange(len(o)) + 1, o, width=0.8, color=np.where(o >= 0, C_BAD, C_DATA))
    a1.axhline(0, color="k", lw=0.8)
    lim = max(1.0, np.abs(o).max() * 1.1)
    a1.set_ylim(-lim, lim); a1.set_ylabel("output $o$"); a1.set_xlabel("pixel $i$")
    plt.tight_layout(); plt.show()

sl = dict(min=-2.0, max=2.0, step=0.25, continuous_update=False)
interact(kernel_1d, w0=FloatSlider(value=-0.5, **sl), w1=FloatSlider(value=0.0, **sl),
         w2=FloatSlider(value=0.5, **sl));


### 1.2 Two dimensions

An image is a 2D grid, so the kernel becomes a small 2D array $K$ and it slides in both directions.
At each position $(i, j)$:

$$\boxed{\;O(i,j) \;=\; \sum_{m=0}^{2}\ \sum_{n=0}^{2}\ K(m,n)\; I(i+m,\; j+n)\;}$$

| Symbol | Meaning |
|---|---|
| $I$ | the input image, one number per pixel |
| $K$ | the kernel, here $3 \times 3$ |
| $O$ | the output, or feature map |
| $(i, j)$ | row and column of the window's top-left corner |
| $(m, n)$ | row and column inside the window |

Each output value takes nine multiplications and one sum.

The 2D stencils you know from finite differences are $3 \times 3$ kernels. With $h = 1$:

$$
K_{\nabla^2} = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}
\qquad
K_{\partial/\partial x} = \begin{bmatrix} 0 & 0 & 0 \\ -\tfrac12 & 0 & \tfrac12 \\ 0 & 0 & 0 \end{bmatrix}
\qquad
K_{\text{Sobel}} = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}
$$

The first is the five-point Laplacian. The second is the central difference in $x$. The third, the
**Sobel kernel**, is the difference in $x$ averaged over three rows with weights 1, 2, 1. It is
the standard edge detector in image processing, and it is used below.

To keep the arithmetic readable, the next few cells use a synthetic image with the same look as the
data: discs of fibre in a matrix, periodic across the edges. It is shown beside a real one, with a
small crop taken from it.

In [ ]:
# --- A synthetic microstructure, and a small crop of it -----------------------
def synth_microstructure(n=64, discs=((16, 16, 7), (17, 44, 9), (46, 22, 8),
                                      (44, 50, 6), (32, 32, 5), (58, 60, 7))):
    yy, xx = np.mgrid[0:n, 0:n]
    img = np.zeros((n, n), dtype=np.float32)
    for cy, cx, r in discs:
        dy = np.minimum(np.abs(yy - cy), n - np.abs(yy - cy))   # periodic distance
        dx = np.minimum(np.abs(xx - cx), n - np.abs(xx - cx))
        img[dy**2 + dx**2 <= r*r] = 1.0
    return img

SYN = synth_microstructure()
SMALL = SYN[8:32, 8:32].reshape(12, 2, 12, 2).mean(axis=(1, 3)).round(1).astype(np.float32)
K_SOBEL = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
K_LAP   = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
axes[0].imshow(SYN, cmap="gray", interpolation="nearest")
axes[0].set_title(f"synthetic image, 64x64\nfibre fraction {SYN.mean():.3f}", fontsize=9)
axes[1].imshow(X_IMG[0], cmap="gray", interpolation="nearest")
axes[1].set_title(f"a real microstructure\nfibre fraction {X_IMG[0].mean():.3f}", fontsize=9)
axes[2].imshow(SMALL, cmap="gray", interpolation="nearest", vmin=0, vmax=1)
axes[2].set_title("12x12 crop used below\n(block-averaged, so edges are grey)", fontsize=9)
for a in axes:
    bare(a)
plt.tight_layout(); plt.show()


### Animation 2: the kernel sweeping across an image

The Sobel kernel slides over an 8x8 piece of the crop, one position per frame, and every one of
the 36 positions is shown.

**Left:** the input, with the current $3 \times 3$ window. **Middle:** the nine input values under
the window in black, each with the kernel weight it is multiplied by in orange, and the sum
underneath. **Right:** the feature map filling in. Grey cells have not been computed yet.

Keep an eye on the orange numbers in the middle panel: they stay the same in every frame.

In [ ]:
# --- Animation 2: the kernel sweeping across the input --------------------------
IN_A = SMALL[1:9, 0:8]
OH, OW = IN_A.shape[0] - 2, IN_A.shape[1] - 2
POS = [(i, j) for i in range(OH) for j in range(OW)]
FULL_A = conv2d_valid(IN_A, K_SOBEL)

fig, (axL, axM, axR) = plt.subplots(1, 3, figsize=(13, 4.6), dpi=ANIM_DPI)
axL.imshow(IN_A, cmap="gray", interpolation="nearest", vmin=0, vmax=1)
for g in range(IN_A.shape[0] + 1):
    axL.axhline(g - 0.5, color="0.55", lw=0.5); axL.axvline(g - 0.5, color="0.55", lw=0.5)
bare(axL)
box = Rectangle((-0.5, -0.5), 3, 3, fill=False, ec=C_FIT, lw=3.0, zorder=5)
axL.add_patch(box)

axM.set_xlim(-0.75, 3.05); axM.set_ylim(3.6, -0.55); bare(axM)
txt_I = []
for r in range(3):
    for c in range(3):
        axM.add_patch(Rectangle((c, r), 1, 1, fc="0.965", ec="0.55", lw=1.0))
        txt_I.append(axM.text(c + 0.5, r + 0.34, "", ha="center", va="center", fontsize=15))
        axM.text(c + 0.5, r + 0.72, f"x {K_SOBEL[r, c]:+g}", ha="center", va="center",
                 fontsize=11, color=C_FIT)
sum_txt2 = axM.text(1.5, 3.35, "", ha="center", va="center", fontsize=13, color=C_FIT)
axM.set_title("input values (black) x kernel weights (orange)", fontsize=10)

gR = number_grid(axR, (OH, OW), vmax=np.abs(FULL_A).max(), title="feature map $O$", fontsize=9)

def frame_2d(f):
    part = np.full((OH, OW), np.nan)
    for k in range(f + 1):
        part[POS[k]] = FULL_A[POS[k]]
    fill_grid(gR, part, "{:+.1f}")
    i, j = POS[f]
    box.set_xy((j - 0.5, i - 0.5))
    for t, v in zip(txt_I, IN_A[i:i+3, j:j+3].ravel()):
        t.set_text(f"{v:g}")
    sum_txt2.set_text(f"O({i},{j}) = {FULL_A[i, j]:+.1f}")
    axL.set_title(f"input $I$, window at (i, j) = ({i}, {j})", fontsize=10)
    for p in list(axR.patches):
        p.remove()
    outline(axR, i, j)
    return []

anim = animation.FuncAnimation(fig, frame_2d, frames=len(POS), interval=300)
play(anim, fig)


**What the animation shows.** The same nine orange weights are used at all 36 positions. Only the
black input values change. This is called **weight sharing**, and Part 2 shows why it matters.

The feature map is positive on one side of the fibre and negative on the other, and close to zero
where the input is uniform. The weights of the Sobel kernel add up to zero, so any flat patch
cancels. Only the boundaries survive, and the sign says which way the value was changing, just as
in 1D.

### 1.3 What different kernels pick out

The gallery applies six fixed kernels to the full synthetic image. Each one picks out something
different. Red is positive output, blue is negative.

In [ ]:
# --- A gallery of fixed kernels ----------------------------------------------
KERNELS = {
    "identity":             np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], np.float32),
    "average (blur)":       np.ones((3, 3), np.float32) / 9.0,
    "Sobel, x direction":   K_SOBEL,
    "Sobel, y direction":   K_SOBEL.T.copy(),
    "Laplacian stencil":    K_LAP,
    "diagonal":             np.array([[-2, -1, 0], [-1, 0, 1], [0, 1, 2]], np.float32),
}

fig, axes = plt.subplots(2, 4, figsize=(13, 6.2))
axes = axes.ravel()
axes[0].imshow(SYN, cmap="gray", interpolation="nearest")
axes[0].set_title("input", fontsize=9)
for ax, (name, k) in zip(axes[1:7], KERNELS.items()):
    out = conv2d_valid(SYN, k)
    m = np.abs(out).max() + 1e-9
    ax.imshow(out, cmap="coolwarm", vmin=-m, vmax=m, interpolation="nearest")
    ax.set_title(f"{name}\nkernel sum = {k.sum():g}", fontsize=9)
axes[7].axis("off")
for a in axes[:7]:
    bare(a)
plt.tight_layout(); plt.show()


**What the gallery shows.** The two Sobel kernels light up the boundaries that run across
their direction: the $x$ kernel finds the left and right edges of each disc, the $y$ kernel the
top and bottom edges. The Laplacian finds every boundary, whatever its direction, as a thin
positive and negative ring. The average kernel only blurs.

Look at the kernel sums in the titles. A kernel whose weights add to zero ignores flat regions and
reports change. A kernel whose weights add to one keeps the local level.

### Try it: write your own 2D kernel

Type nine numbers, row by row, separated by commas or spaces. The preset menu fills the box for you.

In [ ]:
# --- Interactive kernel editor --------------------------------------------------
PRESETS = {"custom": None,
           "Sobel x":   "-1,0,1, -2,0,2, -1,0,1",
           "Sobel y":   "-1,-2,-1, 0,0,0, 1,2,1",
           "Laplacian": "0,1,0, 1,-4,1, 0,1,0",
           "average":   "0.111,0.111,0.111, 0.111,0.111,0.111, 0.111,0.111,0.111",
           "sharpen":   "0,-1,0, -1,5,-1, 0,-1,0"}

def kernel_editor(preset="Sobel x", values="1,0,-1, 2,0,-2, 1,0,-1"):
    text = PRESETS[preset] if PRESETS[preset] is not None else values
    try:
        nums = [float(v) for v in text.replace(",", " ").split()]
        if len(nums) != 9:
            raise ValueError
    except ValueError:
        print("Give exactly nine numbers, separated by commas or spaces.")
        return
    k = np.array(nums, dtype=np.float32).reshape(3, 3)
    out = conv2d_valid(SYN, k)
    m = np.abs(out).max() + 1e-9
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.9), gridspec_kw={"width_ratios": [1, 0.6, 1]})
    axes[0].imshow(SYN, cmap="gray", interpolation="nearest"); axes[0].set_title("input", fontsize=10)
    kk = np.abs(k).max() + 1e-9
    axes[1].imshow(k, cmap="coolwarm", vmin=-kk, vmax=kk, interpolation="nearest")
    for r in range(3):
        for c in range(3):
            axes[1].text(c, r, f"{k[r, c]:g}", ha="center", va="center", fontsize=12)
    axes[1].set_title(f"kernel, sum = {k.sum():+.3g}", fontsize=10)
    axes[2].imshow(out, cmap="coolwarm", vmin=-m, vmax=m, interpolation="nearest")
    axes[2].set_title(f"feature map, range {out.min():+.2f} to {out.max():+.2f}", fontsize=10)
    for a in axes:
        bare(a)
    plt.tight_layout(); plt.show()

interact(kernel_editor, preset=Dropdown(options=list(PRESETS.keys()), value="Sobel x"),
         values=Text(value="1,0,-1, 2,0,-2, 1,0,-1", description="custom:", continuous_update=False));


Choose "custom" and try the Sobel kernel with every sign flipped (the default text). The
feature map swaps colour. Then try nine random numbers. The output still varies across the image,
but it is hard to say what the kernel detects. Training a CNN starts from random kernels like
these.

### Summary of Part 1

- A convolution slides a small kernel over the input and writes one weighted sum per position.
- Finite-difference stencils are convolution kernels. So are blurs and edge detectors.
- The kernel decides what the output reports. A CNN **learns** its kernels, instead of having
  them written down by a person.

---

# Part 2 - From a neuron to a convolution layer

### 2.1 Start from the neuron of Notebook 2

A neuron takes inputs $x_1, \dots, x_d$, forms a weighted sum, adds a bias, and applies an
activation function:

$$z = \sum_{p=1}^{d} w_p\, x_p + b, \qquad a = \phi(z)$$

To feed an image to a neuron, list its pixels as $x_1, \dots, x_d$. A layer of such neurons, each
with its own weights, is a **dense layer**, written with a weight matrix $W$ (one row per neuron):

$$\mathbf{z} = W\mathbf{x} + \mathbf{b}$$

Take a tiny 6x6 image, so $d = 36$, and a layer of 16 neurons. $W$ is $16 \times 36$, and with
the biases the layer has $16 \times 36 + 16 = 592$ numbers to learn.

Now make two changes, one at a time.

1. **Local connections.** Neuron number $p$ may only look at one $3 \times 3$ patch of the image,
   and each neuron gets a different patch. Every row of $W$ now has 9 non-zero entries instead of 36.
   Parameters: $16 \times 9 + 16 = 160$.
2. **Shared weights.** All 16 neurons use the **same** nine weights and the same bias. Parameters:
   $9 + 1 = 10$.

After the second change, the 16 neurons compute the same 16 values of a convolution with a
$3 \times 3$ kernel on the 6x6 image. **A convolution layer is a dense layer in which most weights
are forced to zero and the rest are forced to be equal.** Each pixel of the feature map is the
output of one neuron.

### Animation 3: a dense layer becomes a convolution layer

**Left:** the 36 weights that one neuron applies to the 6x6 image. Grey means "not connected".
**Middle:** the 16 neurons, laid out as a 4x4 output. The orange box is the neuron shown on the left.
**Right:** the full weight matrix $W$, one row per neuron and one column per pixel, with that
neuron's row outlined.

The animation runs through the three stages: dense, local, shared. In each stage it steps through a
few neurons. The right panel shows the change most clearly. In the dense stage every row is different and full. In the local
stage each row is a short band. In the shared stage every band holds the same nine colours, shifted
along.

In [ ]:
# --- Animation 3: dense -> local -> shared ---------------------------------------
rng = np.random.default_rng(3)
X6 = X_IMG[0, 45:51, 6:12]                        # a 6x6 piece of a real image
K_DEMO = np.round(rng.normal(size=(3, 3)), 1)     # an arbitrary 3x3 kernel with no zeros
K_DEMO[K_DEMO == 0] = 0.1
POS6 = [(i, j) for i in range(4) for j in range(4)]

def window_mask(p):
    i, j = POS6[p]
    m = np.zeros((6, 6)); m[i:i+3, j:j+3] = 1
    return m.ravel()

W_DENSE  = rng.normal(size=(16, 36))
W_LOCAL  = np.array([W_DENSE[p] * window_mask(p) for p in range(16)])
W_SHARED = np.zeros((16, 36))
for p, (i, j) in enumerate(POS6):
    w = np.zeros((6, 6)); w[i:i+3, j:j+3] = K_DEMO
    W_SHARED[p] = w.ravel()

STAGES = [("1. dense: every neuron sees every pixel, all weights different", W_DENSE, 16*36 + 16),
          ("2. local: each neuron sees one 3x3 patch, weights still different", W_LOCAL, 16*9 + 16),
          ("3. shared: every neuron uses the same 9 weights. This is a convolution", W_SHARED, 9 + 1)]
SHOW = [0, 1, 5, 10, 15, 15, 15]
FR3 = [(s, p) for s in range(3) for p in SHOW]
vm = 2.0

fig = plt.figure(figsize=(14, 5.4), dpi=ANIM_DPI)
gsp = fig.add_gridspec(1, 3, width_ratios=[1.0, 0.75, 2.2], wspace=0.25, top=0.78, bottom=0.08)
gL = number_grid(fig.add_subplot(gsp[0]), (6, 6), vmax=vm, title="weights of one neuron", fontsize=8)
gC = number_grid(fig.add_subplot(gsp[1]), (4, 4), cmap="Greys", vmax=1, title="16 neurons = 4x4 output")
axW = fig.add_subplot(gsp[2])
cmW = plt.get_cmap("coolwarm").copy(); cmW.set_bad("0.94")
imW = axW.imshow(np.full((16, 36), np.nan), cmap=cmW, vmin=-vm, vmax=vm, aspect="auto",
                 interpolation="nearest")
axW.set_xlabel("pixel (column of W), 36 of them"); axW.set_ylabel("neuron (row of W)")
axW.set_xticks(range(0, 36, 6)); axW.set_yticks(range(0, 16, 3)); axW.grid(False)
axW.set_title("weight matrix $W$, 16 x 36", fontsize=10)
head = fig.text(0.02, 0.93, "", fontsize=13, weight="bold")
cnt = fig.text(0.02, 0.87, "", fontsize=12, color=C_FIT)
marks = []

def frame_w(f):
    s, p = FR3[f]
    name, W, npar = STAGES[s]
    while marks:
        marks.pop().remove()
    Wm = np.where(W == 0, np.nan, W)
    fill_grid(gL, Wm[p].reshape(6, 6), "{:+.1f}")
    fill_grid(gC, np.full((4, 4), np.nan))
    imW.set_data(Wm)
    i, j = POS6[p]
    marks.append(outline(gL["ax"], i, j, 3, 3))
    marks.append(outline(gC["ax"], i, j))
    marks.append(axW.add_patch(Rectangle((-0.5, p - 0.5), 36, 1, fill=False, ec=C_FIT, lw=2.5)))
    gL["ax"].set_title(f"weights of neuron {p}", fontsize=10)
    head.set_text(name)
    cnt.set_text(f"numbers to learn: {npar}")
    return []

anim = animation.FuncAnimation(fig, frame_w, frames=len(FR3), interval=800)
play(anim, fig)


**What the animation shows.** In the dense stage the right panel is full: 576 different weights.
In the local stage each row keeps only a short band, but every band is still different. In the
shared stage every row carries the same nine colours, each row moved along by one pixel, with a jump
of three pixels at the start of each new output row. The count falls from 592 to 160 to 10.

The next cell checks the claim with numbers: the shared matrix times the flattened image gives
the same 16 values as the convolution loop from Part 1.

In [ ]:
# --- Check: the shared weight matrix IS the convolution -------------------------
b_demo = 0.3
via_matrix = (W_SHARED @ X6.ravel() + b_demo).reshape(4, 4)
via_conv   = conv2d_valid(X6, K_DEMO.astype(np.float32)) + b_demo
print("W x + b, reshaped to 4x4:")
print(np.round(via_matrix, 2))
print("convolution + b:")
print(np.round(via_conv, 2))
print(f"largest difference: {np.abs(via_matrix - via_conv).max():.1e}")
print()
for name, W, npar in STAGES:
    print(f"{name[:9]:10s} non-zero weights {int((W != 0).sum()):4d}   numbers to learn {npar:4d}")


### 2.2 Bias and activation: turning a feature map into a detector

A convolution layer also has a bias and an activation, like a neuron. With the ReLU
activation of Notebook 2, $\mathrm{ReLU}(z) = \max(0, z)$, the layer output is

$$\boxed{\;A(i,j) \;=\; \mathrm{ReLU}\Big(\sum_{m}\sum_{n} K(m,n)\, I(i+m, j+n) \;+\; b\Big)\;}$$

The bias works as a **threshold**. With $b = -2$, a position only produces an output if the
weighted sum is larger than 2. Everything weaker is set to zero. The ReLU also throws away one
sign: a Sobel-$x$ unit with ReLU keeps the left edges of the fibres and discards the right edges.
A network that needs both kinds of edge needs two kernels, one of each sign.

Move the bias slider and watch how much of the image survives.

In [ ]:
# --- Interactive: bias as a threshold -------------------------------------------
def detector(kernel="Sobel x", b=-2.0):
    k = {"Sobel x": K_SOBEL, "Sobel y": K_SOBEL.T.copy(), "Laplacian": K_LAP}[kernel]
    O = conv2d_valid(SYN, k) + b
    A = np.maximum(O, 0)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    axes[0].imshow(SYN, cmap="gray", interpolation="nearest"); axes[0].set_title("input", fontsize=10)
    m = np.abs(O).max()
    axes[1].imshow(O, cmap="coolwarm", vmin=-m, vmax=m, interpolation="nearest")
    axes[1].set_title(f"weighted sum + b,  b = {b:+.1f}", fontsize=10)
    axes[2].imshow(A, cmap="magma", interpolation="nearest", vmin=0, vmax=max(A.max(), 1e-6))
    axes[2].set_title(f"after ReLU: {100 * (A > 0).mean():.1f}% of units active", fontsize=10)
    for a in axes:
        bare(a)
    plt.tight_layout(); plt.show()

interact(detector, kernel=Dropdown(options=["Sobel x", "Sobel y", "Laplacian"], value="Sobel x"),
         b=FloatSlider(value=-2.0, min=-6.0, max=2.0, step=0.5, continuous_update=False));


### 2.3 Why sharing matters: the parameter count

For a dense layer with $n_{in}$ inputs and $n_{out}$ outputs, Notebook 2 gave

$$P_{\text{dense}} = n_{in}\, n_{out} + n_{out}$$

For a convolution layer with a $k \times k$ kernel, and $c_{out}$ different kernels (Part 3
explains why you want more than one):

$$\boxed{\;P_{\text{conv}} = k^2\, c_{out} + c_{out}\;}$$

The image size does not appear. A dense layer grows with the number of pixels. A convolution
layer grows only with the size and number of the patterns it looks for.

In [ ]:
# --- Parameters: dense versus convolutional --------------------------------------
sizes = np.array([16, 32, 64, 128, 256])
n_hidden, n_filters, k = 256, 16, 3
dense_p = sizes**2 * n_hidden + n_hidden
conv_p  = np.full(len(sizes), k**2 * n_filters + n_filters)

print(f"{'image':>9s} {'dense layer, 256 units':>24s} {'conv layer, 16 kernels 3x3':>28s}")
for s, d, c in zip(sizes, dense_p, conv_p):
    print(f"{s:>4d}x{s:<4d} {d:>24,d} {c:>28,d}")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(sizes, dense_p, "-o", color=C_BAD, lw=2, label="dense layer, 256 units")
ax.plot(sizes, conv_p, "-s", color=C_DATA, lw=2, label="convolution layer, 16 kernels of 3x3")
ax.axvline(64, color="k", ls=":", lw=1.2); ax.text(68, 400, "our images", fontsize=8)
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("image side (pixels)"); ax.set_ylabel("numbers to learn")
ax.legend(fontsize=8); ax.set_title("the convolution layer does not grow with the image", fontsize=10)
plt.tight_layout(); plt.show()


**What the plot shows.** The red line grows with the square of the image side. At 64x64 the dense
layer already needs about a million numbers. That one layer is where almost all of the
1,065,345 numbers of the flattened network in Notebook 2 sat.
The blue line is flat at 160.

There is a second benefit that the count does not show. A dense layer must learn what a fibre
boundary looks like separately at every pixel position. A convolution layer uses one kernel
everywhere, so every boundary in every training image teaches the same nine weights.

### Summary of Part 2

- A convolution layer is a layer of neurons with **local connections** and **shared weights**.
- Each pixel of the feature map is one neuron's output. The bias acts as a threshold, the ReLU
  keeps only strong responses of one sign.
- The number of weights depends on the kernel size, not on the image size.

---

# Part 3 - The other building blocks

A CNN is built from a small set of parts. Part 1 covered the convolution. This part adds the rest,
one at a time, each with its own picture.

### 3.1 Padding and stride

**Padding.** Without it, a $3 \times 3$ kernel cannot be centred on the border pixels, so the output
is two pixels smaller than the input in each direction. Padding adds $p$ extra rows and columns
around the input first, so the border pixels get a full window.

**Stride.** The window does not have to move one pixel at a time. With stride $s$ it jumps $s$
pixels, and the output is roughly $s$ times smaller in each direction.

For an input of side $n$, kernel side $k$, padding $p$ and stride $s$, the output side is

$$\boxed{\;n_{out} \;=\; \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1\;}$$

The floor $\lfloor\ \rfloor$ rounds down: a window that would hang over the edge is not used.
Three cases cover most networks: $k=3, p=0, s=1$ loses two pixels; $k=3, p=1, s=1$ keeps the
size; $s=2$ halves it.

### Animation 4: three settings on the same 5x5 input

**Top row:** the input for each setting. Light cells with a dashed border are padding. The orange
box is the current window. **Bottom row:** the output filling in. Each column finishes at its own
pace and then waits.

Before playing, use the formula to predict the three output sizes.

In [ ]:
# --- Animation 4: padding and stride --------------------------------------------
IN5 = X_IMG[0, 46:51, 7:12]
SETTINGS = [(0, 1), (1, 1), (1, 2)]                  # (padding, stride)

def conv_ps(img, k, p, s):
    padded = np.pad(img, p, mode="constant")
    return conv2d_valid(padded, k)[::s, ::s], padded

panels = []
fig, axes = plt.subplots(2, 3, figsize=(13, 8.2), dpi=ANIM_DPI,
                         gridspec_kw={"height_ratios": [1.35, 1.0]})
for c, (p, s) in enumerate(SETTINGS):
    out, padded = conv_ps(IN5, K_SOBEL, p, s)
    n_out = (5 + 2*p - 3) // s + 1
    pos = [(i, j) for i in range(n_out) for j in range(n_out)]
    gi = number_grid(axes[0, c], padded.shape, cmap="gray", vmax=1.6,
                     title=f"padding p = {p}, stride s = {s}", fontsize=11)
    fill_grid(gi, padded)
    for r in range(padded.shape[0]):
        for q in range(padded.shape[1]):
            if r < p or q < p or r >= p + 5 or q >= p + 5:
                axes[0, c].add_patch(Rectangle((q - 0.5, r - 0.5), 1, 1, fc="#FBF3E4",
                                               ec="0.6", ls="--", lw=0.8, zorder=2))
                gi["tx"][r][q].set_zorder(3)
    go = number_grid(axes[1, c], (n_out, n_out), vmax=4.5, fontsize=11,
                     title=f"output {n_out} x {n_out} = floor(({5}+{2*p}-3)/{s})+1")
    win = Rectangle((-0.5, -0.5), 3, 3, fill=False, ec=C_FIT, lw=3.5, zorder=7)
    axes[0, c].add_patch(win)
    panels.append(dict(out=out, pos=pos, go=go, win=win, s=s, cur=[]))

NF4 = max(len(P["pos"]) for P in panels) + 3

def frame_ps(f):
    for P in panels:
        k = min(f, len(P["pos"]) - 1)
        part = np.full(P["out"].shape, np.nan)
        for t in range(k + 1):
            part[P["pos"][t]] = P["out"][P["pos"][t]]
        fill_grid(P["go"], part, "{:+g}")
        i, j = P["pos"][k]
        P["win"].set_xy((j * P["s"] - 0.5, i * P["s"] - 0.5))
        while P["cur"]:
            P["cur"].pop().remove()
        if f < len(P["pos"]):
            outline(P["go"]["ax"], i, j, store=P["cur"])
    return []

anim = animation.FuncAnimation(fig, frame_ps, frames=NF4, interval=450)
play(anim, fig)


**What the animation shows.** Left: no padding, and the 5x5 input gives a 3x3 output. Middle: one
ring of zeros is added, the window can now sit on every original pixel, and the output is 5x5, the
same as the input. Right: the same padding, but the window jumps two cells at a time, so it visits
only 9 positions and the output is 3x3 again.

Notice the middle panel's border outputs. They are computed partly from padding values that are
not part of the material. What to put in the padding is a modelling choice, and the next cell makes
it for our data.

### 3.2 Periodic padding is a periodic boundary condition

The microstructures in this dataset are **periodic unit cells**. The finite element model that gave
their stiffness used periodic boundary conditions: a fibre leaving the right edge comes back in on
the left. Zero padding would put a fake strip of matrix around the cell. **Circular padding** wraps
the image round, so the kernel sees the neighbouring cell, which is what is physically there. In PyTorch this is
`padding_mode="circular"`, and Notebook 3b uses it.

In [ ]:
# --- Zero padding versus circular padding on a real cell --------------------------
P_SHOW = 10
img = X_IMG[IMG_DEMO]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, mode, title in [(axes[0], "constant", "zero padding: a fake frame of matrix"),
                        (axes[1], "wrap", "circular padding: the neighbouring cells")]:
    ax.imshow(np.pad(img, P_SHOW, mode=mode), cmap="gray", interpolation="nearest")
    ax.add_patch(Rectangle((P_SHOW - 0.5, P_SHOW - 0.5), 64, 64, fill=False, ec=C_FIT, lw=2.5))
    ax.set_title(title + f"\n(orange box = the 64x64 cell, {P_SHOW} px of padding shown)", fontsize=10)
    bare(ax)
plt.tight_layout(); plt.show()

print("Output sizes from the formula, checked against PyTorch on a 64x64 input:")
t_in = torch.zeros(1, 1, 64, 64)
print(f"{'k':>3s} {'p':>3s} {'s':>3s} {'formula':>8s} {'torch':>6s}")
for (k_, p_, s_) in [(3, 0, 1), (3, 1, 1), (5, 2, 1), (3, 1, 2), (7, 3, 2)]:
    formula = (64 + 2 * p_ - k_) // s_ + 1
    got = nn.Conv2d(1, 1, k_, padding=p_, stride=s_, padding_mode="circular")(t_in).shape[-1]
    print(f"{k_:>3d} {p_:>3d} {s_:>3d} {formula:>8d} {got:>6d}")


**What the figure shows.** On the left, fibres cut by the cell edge end abruptly at the orange line.
A kernel sitting there would report a boundary that does not exist in the material. On the right,
the same fibres continue across the edge, because the padding is the cell repeated. The table checks
the size formula against PyTorch.

### 3.3 Several kernels, several channels

One kernel finds one kind of feature. A layer normally has many kernels, say 16. Each produces its
own feature map, and the 16 maps are stacked. Each map in the stack is called a **channel**. A
colour photograph also has channels: red, green and blue.

The next layer then receives a stack of $c_{in}$ channels. Its kernel is a stack as well, one
$3 \times 3$ slice per input channel, and the output is the sum over all channels:

$$\boxed{\;O(i,j) \;=\; \sum_{c=1}^{c_{in}} \sum_{m}\sum_{n} K_c(m,n)\; I_c(i+m,\; j+n) \;+\; b\;}$$

A layer with $c_{out}$ such kernels therefore has

$$\boxed{\;P_{\text{conv}} \;=\; k^2\, c_{in}\, c_{out} \;+\; c_{out}\;}$$

numbers to learn. With $c_{in} = 1$ this is the formula of Part 2.

In the animation, the input has two channels, and both come from Part 1. Channel 1 is the Sobel-$x$
map after ReLU: it marks left edges of fibres. Channel 2 is the Sobel-$y$ map after ReLU: top edges.
The second-layer kernel has one slice for each, and both slices here are the same plus-shaped
stencil. The output is large wherever there is a left edge, a top edge, or both. The second layer
has combined two simple detectors into a new one.

In [ ]:
# --- Animation 5: a kernel with two input channels --------------------------------
FX = np.maximum(conv2d_valid(SYN, K_SOBEL), 0)          # channel 1: left edges
FY = np.maximum(conv2d_valid(SYN, K_SOBEL.T.copy()), 0)  # channel 2: top edges
CH = np.stack([FX[7:13, 34:40], FY[7:13, 34:40]])        # a 6x6 piece of each
K_PLUS = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], np.float32)
K2 = np.stack([K_PLUS, K_PLUS])                          # one slice per input channel
OUT2 = sum(conv2d_valid(CH[c], K2[c]) for c in range(2))
POS2 = [(i, j) for i in range(4) for j in range(4)]

fig = plt.figure(figsize=(13, 7.2), dpi=ANIM_DPI)
gsp = fig.add_gridspec(2, 3, width_ratios=[1.3, 0.8, 1.1], hspace=0.3, wspace=0.25,
                       top=0.86, bottom=0.04)
gI = [number_grid(fig.add_subplot(gsp[c, 0]), (6, 6), cmap="Blues", vmax=4.5,
                  title=t, fontsize=10)
      for c, t in enumerate(["input channel 1: left edges", "input channel 2: top edges"])]
gK = [number_grid(fig.add_subplot(gsp[c, 1]), (3, 3), cmap="Oranges", vmax=2.5,
                  title=f"kernel slice $K_{c+1}$", fontsize=12) for c in range(2)]
for c in range(2):
    fill_grid(gI[c], CH[c]); fill_grid(gK[c], K2[c])
gO = number_grid(fig.add_subplot(gsp[:, 2]), (4, 4), cmap="Purples", vmax=40,
                 title="output (one channel)", fontsize=12)
eq = fig.text(0.02, 0.93, "", fontsize=13)
marks = []

def frame_ch(f):
    while marks:
        marks.pop().remove()
    i, j = POS2[f]
    parts = [float((CH[c, i:i+3, j:j+3] * K2[c]).sum()) for c in range(2)]
    so_far = np.full((4, 4), np.nan)
    for t in range(f + 1):
        so_far[POS2[t]] = OUT2[POS2[t]]
    fill_grid(gO, so_far)
    for c in range(2):
        outline(gI[c]["ax"], i, j, 3, 3, store=marks)
    outline(gO["ax"], i, j, store=marks)
    eq.set_text(f"O({i},{j}) = (channel 1 x K1) + (channel 2 x K2) = {parts[0]:g} + {parts[1]:g} = {OUT2[i, j]:g}")
    return []

anim = animation.FuncAnimation(fig, frame_ch, frames=len(POS2), interval=550)
display(play(anim, fig))

for c_in, c_out in [(2, 1), (1, 16), (16, 32), (32, 64)]:
    layer = nn.Conv2d(c_in, c_out, 3)
    n = sum(p.numel() for p in layer.parameters())
    print(f"c_in = {c_in:2d}, c_out = {c_out:2d}:  k^2 c_in c_out + c_out = {9*c_in*c_out + c_out:6d}"
          f"   PyTorch counts {n:6d}")


**What the animation shows.** The same window is drawn on both input channels at once, because one
output value reads both. The line at the top adds the two contributions. The output is largest
in the diagonal band from bottom left to top right, where the fibre boundary runs and both
channels are non-zero.

Deeper layers in a CNN work in this way. Early layers find simple features, such as edges in one
direction, and later layers combine them. In a trained network these combinations are learned
from the data, not chosen by hand.

### 3.4 Pooling

Pooling shrinks a feature map by replacing each small window with one number. **Max pooling**, with
a $2 \times 2$ window and stride 2, keeps the largest value:

$$\boxed{\;P(i,j) \;=\; \max\big\{A(2i, 2j),\ A(2i, 2j{+}1),\ A(2i{+}1, 2j),\ A(2i{+}1, 2j{+}1)\big\}\;}$$

It has no weights. It halves the size in each direction, and it makes the output less sensitive to
a feature moving by a pixel. For anyone who knows multigrid, it plays the role of restriction to a
coarser grid.

Pooling also has a simple backward rule, which Part 4 needs: only the cell that won receives any
gradient. The animation shows both directions.

In [ ]:
# --- Animation 6: max pooling, forward and backward --------------------------------
TOY = np.array([[3, 1, 2, 8], [5, 6, 4, 2], [9, 1, 3, 7], [4, 8, 2, 5]], dtype=np.float32)
G_IN = np.array([[0.1, -0.2], [0.3, 0.4]])                # gradient arriving from above
WINS = [(0, 0), (0, 1), (1, 0), (1, 1)]
POOLED = np.zeros((2, 2)); ARG = {}
for (p, q) in WINS:
    w = TOY[2*p:2*p+2, 2*q:2*q+2]
    r, c = np.unravel_index(w.argmax(), w.shape)
    ARG[(p, q)] = (2*p + r, 2*q + c); POOLED[p, q] = w.max()

FR6 = [("fwd", k) for k in range(4)] + [("fwd", 3)] + [("bwd", k) for k in range(4)] + [("bwd", 3)] * 2

fig, axes = plt.subplots(2, 2, figsize=(9, 8.4), dpi=ANIM_DPI,
                         gridspec_kw={"width_ratios": [2, 1.2]})
gA  = number_grid(axes[0, 0], (4, 4), cmap="Blues", vmax=10, title="feature map $A$", fontsize=14)
gP  = number_grid(axes[0, 1], (2, 2), cmap="Blues", vmax=10, title="pooled $P$", fontsize=14)
gdA = number_grid(axes[1, 0], (4, 4), cmap="PiYG", vmax=0.5, title=r"gradient $\partial L/\partial A$", fontsize=14)
gdP = number_grid(axes[1, 1], (2, 2), cmap="PiYG", vmax=0.5, title=r"gradient $\partial L/\partial P$", fontsize=14)
fill_grid(gA, TOY)
head6 = fig.text(0.03, 0.96, "", fontsize=13, weight="bold")
marks = []

def frame_pool(f):
    kind, k = FR6[f]
    while marks:
        marks.pop().remove()
    p, q = WINS[k]
    r, c = ARG[(p, q)]
    shown = np.full((2, 2), np.nan)
    for t in range(k + 1 if kind == "fwd" else 4):
        shown[WINS[t]] = POOLED[WINS[t]]
    fill_grid(gP, shown)
    if kind == "fwd":
        head6.set_text(f"FORWARD: keep the largest of the four, max = {POOLED[p, q]:g}")
        head6.set_color(C_DATA)
        fill_grid(gdA, np.full((4, 4), np.nan)); fill_grid(gdP, np.full((2, 2), np.nan))
        outline(gA["ax"], 2*p, 2*q, 2, 2, store=marks)
    else:
        head6.set_text(f"BACKWARD: the gradient {G_IN[p, q]:+g} goes to the winner only, the rest get 0")
        head6.set_color(C_BAD)
        fill_grid(gdP, G_IN)
        dA = np.full((4, 4), np.nan)
        for t in range(k + 1):
            pp, qq = WINS[t]
            dA[2*pp:2*pp+2, 2*qq:2*qq+2] = 0
            dA[ARG[(pp, qq)]] = G_IN[pp, qq]
        fill_grid(gdA, dA, "{:+g}")
        outline(gdA["ax"], 2*p, 2*q, 2, 2, store=marks)
        outline(gdP["ax"], p, q, store=marks)
    outline(gA["ax"], r, c, color=C_ALT, lw=4, store=marks)
    outline(gP["ax"], p, q, store=marks)
    return []

anim = animation.FuncAnimation(fig, frame_pool, frames=len(FR6), interval=1000)
play(anim, fig)


**What the animation shows.** Forward: each orange $2 \times 2$ window sends its largest value,
outlined in green, to one cell of $P$. Backward: each gradient in $\partial L/\partial P$ is passed to
the cell that won that window, and the other three cells get zero. This is the exact derivative: a
small change to a cell that did not win leaves the maximum unchanged.

The next cell checks the other claim, that pooling makes the output less sensitive to small shifts.
It shifts the synthetic image by one pixel and measures how much the feature map changes, before and
after $4 \times 4$ max pooling.

In [ ]:
# --- Pooling and small shifts ----------------------------------------------------
def maxpool(a, size=2):
    h, w = (a.shape[0] // size) * size, (a.shape[1] // size) * size
    return a[:h, :w].reshape(h // size, size, w // size, size).max(axis=(1, 3))

f0 = np.maximum(conv2d_valid(SYN, K_SOBEL), 0)
f1 = np.maximum(conv2d_valid(np.roll(SYN, 1, axis=1), K_SOBEL), 0)   # image moved 1 px right
p0, p1 = maxpool(f0, 4), maxpool(f1, 4)
d_raw  = np.abs(f0 - f1).sum() / f0.sum()
d_pool = np.abs(p0 - p1).sum() / p0.sum()
print("move the image one pixel to the right:")
print(f"  relative change of the feature map          {d_raw:.3f}")
print(f"  relative change after 4x4 max pooling        {d_pool:.3f}")

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, (a, t) in zip(axes, [(f0, "feature map"), (f1, "image moved 1 px"),
                             (p0, "pooled"), (p1, "pooled, image moved 1 px")]):
    ax.imshow(a, cmap="magma", interpolation="nearest")
    ax.set_title(f"{t}\n{a.shape[0]} x {a.shape[1]}", fontsize=9)
    bare(ax)
plt.tight_layout(); plt.show()


**What the numbers show.** Moving the image by a single pixel changes a large fraction of the
feature map, because every edge response moves to a different pixel. After pooling, most of the
winners still fall in the same $4 \times 4$ block, so the change printed on the second line is
smaller. A shift of a whole block still changes the pooled map, so pooling only helps with small
shifts.

### 3.5 Stacking layers: how far can one unit see?

In an explicit time-stepping scheme with a three-point stencil, information travels one node per
step. After $n$ steps, a node depends on $2n + 1$ nodes of the initial condition: its **domain of
dependence** grows with every step.

Stacked convolution layers behave in the same way. The patch of the input that one unit depends on
is its **receptive field**. Write $r_\ell$ for its side after layer $\ell$ and $J_\ell$ for the
spacing between neighbouring units of that layer, measured in input pixels. A layer with kernel side
$k$ and stride $s$ gives

$$\boxed{\;r_\ell = r_{\ell-1} + (k_\ell - 1)\, J_{\ell-1}, \qquad J_\ell = J_{\ell-1}\, s_\ell\;}$$

starting from $r_0 = 1$ and $J_0 = 1$. A $3 \times 3$ convolution adds $2J$. A $2 \times 2$ pool
adds $J$ and doubles the spacing, so every layer after it adds twice as much. This is why the field
grows quickly once pooling is used.

The network used in Notebook 3b is three blocks, each a $3 \times 3$ convolution, a ReLU and a
$2 \times 2$ max pool. The animation follows one unit near the centre of each layer and draws, on a
real microstructure, the patch it can see. The patch is measured rather than taken from the formula:
PyTorch reports which input pixels have a non-zero gradient for that unit. The ReLUs are left out
for this measurement so that no path is switched off.

In [ ]:
# --- Animation 7: the receptive field, layer by layer -----------------------------
LAYERS = [("input", None), ("conv 1", ("c", 3, 1)), ("pool 1", ("p", 2, 2)),
          ("conv 2", ("c", 3, 1)), ("pool 2", ("p", 2, 2)),
          ("conv 3", ("c", 3, 1)), ("pool 3", ("p", 2, 2))]

# the formula
R_F, J_F, SIZE = [1], [1], [64]
for name, spec in LAYERS[1:]:
    kind, k_, s_ = spec
    R_F.append(R_F[-1] + (k_ - 1) * J_F[-1])
    J_F.append(J_F[-1] * s_)
    SIZE.append(SIZE[-1] // s_)

# the measurement: a linear copy of the network with all weights positive
def measured_patch(n_layers):
    torch.manual_seed(SEED)
    mods = []
    for name, spec in LAYERS[1:n_layers + 1]:
        kind, k_, s_ = spec
        if kind == "c":
            cv = nn.Conv2d(1, 1, 3, padding=1, bias=False)
            nn.init.constant_(cv.weight, 1.0)
            mods.append(cv)
        else:
            mods.append(nn.AvgPool2d(2))
    net = nn.Sequential(*mods)
    x = torch.zeros(1, 1, 64, 64, requires_grad=True)
    y = net(x)
    u = y.shape[-1] // 2
    y[0, 0, u, u].backward()
    rows = torch.nonzero(x.grad[0, 0].abs().sum(1)).ravel()
    cols = torch.nonzero(x.grad[0, 0].abs().sum(0)).ravel()
    return int(rows[0]), int(cols[0]), int(rows[-1] - rows[0] + 1)

PATCH = [measured_patch(n) for n in range(len(LAYERS))]
print(f"{'layer':8s} {'map size':>9s} {'spacing J':>10s} {'field, formula':>15s} {'field, measured':>16s}")
for (name, _), sz, J_, r_, pt in zip(LAYERS, SIZE, J_F, R_F, PATCH):
    print(f"{name:8s} {sz:>6d}x{sz:<2d} {J_:>10d} {r_:>15d} {pt[2]:>16d}")

img_rf = X_IMG[IMG_DEMO]
fig, (axI, axR) = plt.subplots(1, 2, figsize=(12, 5.2), dpi=ANIM_DPI,
                               gridspec_kw={"width_ratios": [1, 1.2]})
axI.imshow(img_rf, cmap="gray", interpolation="nearest"); bare(axI)
sq = Rectangle((0, 0), 1, 1, fc=C_FIT, alpha=0.35, ec=C_FIT, lw=2.5)
axI.add_patch(sq)
xs = np.arange(len(LAYERS))
axR.plot(xs, R_F, "o-", color=C_FIT, lw=2, label="receptive field (input pixels)")
axR.plot(xs, SIZE, "s-", color=C_DATA, lw=2, label="feature map side (units)")
cur_r = axR.axvline(0, color="k", lw=1, ls=":")
axR.set_xticks(xs); axR.set_xticklabels([n for n, _ in LAYERS], rotation=30)
axR.set_ylabel("pixels"); axR.set_ylim(0, 68); axR.legend(fontsize=9, loc="center right")

FR7 = [k for k in range(len(LAYERS)) for _ in range(3)] + [len(LAYERS) - 1] * 2

def frame_rf(f):
    k = FR7[f]
    r0, c0, side = PATCH[k]
    sq.set_bounds(c0 - 0.5, r0 - 0.5, side, side)
    cur_r.set_xdata([k, k])
    axI.set_title(f"after {LAYERS[k][0]}: one unit sees {side} x {side} pixels", fontsize=11)
    axR.set_title(f"{LAYERS[k][0]}: map {SIZE[k]} x {SIZE[k]} units, one unit every {J_F[k]} px", fontsize=11)
    return []

anim = animation.FuncAnimation(fig, frame_rf, frames=len(FR7), interval=500)
play(anim, fig)


**What the animation shows.** Left: the orange square is everything in the image that can affect
one unit. Right: the orange line is the side of that square, the blue line is the side of the feature
map. As the network gets deeper the maps get coarser and each unit sees further. The printed table
confirms that the formula and the measurement agree at every layer.

Compare the final square with the fibres in the picture. After the last block a single unit can
see a few fibres and the gaps between them. Notebook 3b shows that this matters: a network whose
units see only a few pixels cannot predict the anisotropy at all.

### 3.6 The whole network, block by block

Here is the complete network used in Notebook 3b:

| Stage | Operation | Output shape (channels x height x width) |
|---|---|---|
| input | | 1 x 64 x 64 |
| block 1 | conv 3x3 with 16 kernels, ReLU, max pool 2x2 | 16 x 32 x 32 |
| block 2 | conv 3x3 with 32 kernels, ReLU, max pool 2x2 | 32 x 16 x 16 |
| block 3 | conv 3x3 with 64 kernels, ReLU, max pool 2x2 | 64 x 8 x 8 |
| flatten | lay the 64 x 8 x 8 numbers out in a list | 4096 |
| dense | 64 neurons, ReLU | 64 |
| output | one neuron, no activation | 1 (a stiffness in GPa) |

The convolution blocks turn the picture into features. The dense layers at the end are the network
of Notebook 2, reading those features instead of pixels.

The animation passes a real microstructure through this network with **random, untrained** weights.
The feature maps therefore do not mean anything yet. Look at the shapes instead: at each block the
maps get smaller and there are more of them.

In [ ]:
# --- Animation 8: the tensor shapes through the network ---------------------------
torch.manual_seed(SEED)
NET = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1, padding_mode="circular"), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1, padding_mode="circular"), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1, padding_mode="circular"), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64 * 8 * 8, 64), nn.ReLU(), nn.Linear(64, 1))

STAGE_AT = {1: ("conv 1 + ReLU", 0), 2: ("pool 1", None), 4: ("conv 2 + ReLU", 3),
            5: ("pool 2", None), 7: ("conv 3 + ReLU", 6), 8: ("pool 3", None),
            9: ("flatten", None), 11: ("dense + ReLU", 10), 12: ("output", 12)}
h = torch.tensor(X_IMG[IMG_DEMO]).view(1, 1, 64, 64)
STAGES8 = [("input", h[0].numpy(), 0)]
with torch.no_grad():
    for idx, layer in enumerate(NET):
        h = layer(h)
        if idx in STAGE_AT:
            name, p_idx = STAGE_AT[idx]
            npar = 0 if p_idx is None else sum(p.numel() for p in NET[p_idx].parameters())
            STAGES8.append((name, h[0].numpy(), npar))

print(f"{'stage':15s} {'shape':>16s} {'numbers stored':>15s} {'weights':>9s}")
for name, a, npar in STAGES8:
    print(f"{name:15s} {str(tuple(a.shape)):>16s} {a.size:>15,d} {npar:>9,d}")
N_TOTAL = sum(p.numel() for p in NET.parameters())
print(f"total weights and biases: {N_TOTAL:,d}")

fig = plt.figure(figsize=(14, 6.4), dpi=ANIM_DPI)
axBk = fig.add_axes([0.02, 0.52, 0.96, 0.44])
axBk.set_xlim(-0.5, len(STAGES8) - 0.5); axBk.set_ylim(-0.1, 1.15); bare(axBk)
blocks = []
for k, (name, a, _) in enumerate(STAGES8):
    if a.ndim == 3:
        c, hh = a.shape[0], a.shape[1]
        side = 0.25 + 0.6 * hh / 64
        layers_drawn = min(int(np.log2(c)) + 1, 7)
        rects = []
        for t in range(layers_drawn):
            rr = Rectangle((k - side / 2 + 0.035 * t, 0.2 + 0.03 * t), side * 0.6, side,
                           fc="0.85", ec="0.4", lw=0.8)
            axBk.add_patch(rr); rects.append(rr)
        label = f"{name}\n{c}x{hh}x{hh}"
    else:
        hgt = 0.25 + 0.6 * min(a.size, 4096) / 4096
        rr = Rectangle((k - 0.06, 0.2), 0.12, hgt, fc="0.85", ec="0.4", lw=0.8)
        axBk.add_patch(rr); rects = [rr]
        label = f"{name}\n{a.size}"
    axBk.text(k, 0.08, label, ha="center", va="top", fontsize=8.5)
    blocks.append(rects)

axMaps = [fig.add_axes([0.03 + 0.118 * t, 0.06, 0.105, 0.3]) for t in range(8)]
axVec = fig.add_axes([0.03, 0.12, 0.93, 0.12])
info = fig.text(0.03, 0.40, "", fontsize=12)

FR8 = [k for k in range(len(STAGES8)) for _ in range(3)]

def frame_net(f):
    k = FR8[f]
    for kk, rects in enumerate(blocks):
        for rr in rects:
            rr.set_facecolor(C_FIT if kk == k else ("0.7" if kk < k else "0.9"))
    name, a, npar = STAGES8[k]
    for t, ax in enumerate(axMaps):
        ax.clear(); bare(ax)
        show = a.ndim == 3 and t < a.shape[0]
        ax.set_visible(show)
        if show:
            ax.imshow(a[t], cmap="viridis" if k else "gray", interpolation="nearest")
            ax.set_title(f"channel {t}", fontsize=8)
    axVec.clear(); bare(axVec)
    axVec.set_visible(a.ndim == 1)
    if a.ndim == 1:
        n_show = min(a.size, 512)
        axVec.imshow(a[None, :n_show], aspect="auto", cmap="viridis", interpolation="nearest")
        axVec.set_title(f"the first {n_show} of {a.size} numbers, laid out in a line", fontsize=9)
    shape_txt = " x ".join(str(s) for s in a.shape)
    info.set_text(f"{name}:  shape {shape_txt},  {a.size:,d} numbers stored,  {npar:,d} weights in this step")
    return []

anim = animation.FuncAnimation(fig, frame_net, frames=len(FR8), interval=450)
play(anim, fig)


**What the animation shows.** Top: the stages of the network. The orange block is the current one,
and its drawing gets smaller as the maps shrink and thicker as the channels increase. Bottom: up to
eight channels at that stage. After pool 3 each map is 8 by 8, one value for each 8 by 8 pixel
block of the original cell. At flatten the spatial layout is given up and the numbers become a list.

The printed table gives the count at every step. Most of the weights sit in the first dense layer,
not in the convolutions that do the pattern finding, because the convolutions share their weights.

### Summary of Part 3

- **Padding** keeps the size, and circular padding matches a periodic cell.
- **Stride** and **pooling** shrink the maps. Pooling has no weights and passes gradient to the
  winner only.
- A layer has many **channels**, and a kernel reads all input channels at once.
- Stacking layers grows the **receptive field**, fastest after pooling.
- A CNN is convolution blocks that extract features, followed by a small dense network.

---

# Part 4 - How a convolution learns its weights

### 4.1 What training needs

Training works as in Notebooks 1 and 2. A loss $L$ measures how wrong the prediction is, and
gradient descent nudges every weight downhill:

$$K(m,n) \;\leftarrow\; K(m,n) - \eta\,\frac{\partial L}{\partial K(m,n)}, \qquad b \;\leftarrow\; b - \eta\,\frac{\partial L}{\partial b}$$

So each convolution layer needs three things from the backward pass:

1. $\partial L/\partial K$, to update its nine weights,
2. $\partial L/\partial b$, to update its bias,
3. $\partial L/\partial I$, the gradient with respect to its own input, to hand on to the layer before it.

### 4.2 The three rules

Suppose the layer below has already been given $\partial L/\partial O(i,j)$ for every output
position, which says how much each output value affects the loss. Each weight $K(m,n)$ was used at
every position, so its gradient collects a contribution from every position:

$$\boxed{\;\frac{\partial L}{\partial K(m,n)} \;=\; \sum_{i}\sum_{j} \frac{\partial L}{\partial O(i,j)}\; I(i+m,\; j+n)\;}$$

In words: slide the window over the input again, multiply each patch by how much its output
mattered, and add the patches up. **The kernel gradient is itself a convolution**: the input,
convolved with the map of output gradients.

The bias is added at every position, so

$$\frac{\partial L}{\partial b} \;=\; \sum_{i}\sum_{j} \frac{\partial L}{\partial O(i,j)}$$

The input gradient: pixel $I(p,q)$ was read by every window that covered it, so each window sends
its kernel, scaled by its own output gradient, back onto its own patch, and the overlaps add:

$$\frac{\partial L}{\partial I(p,q)} \;=\; \sum_{i}\sum_{j} \frac{\partial L}{\partial O(i,j)}\; K(p-i,\; q-j)$$

where only the terms with $0 \le p-i \le 2$ and $0 \le q-j \le 2$ exist. In the matrix picture of
Part 2 this is $\partial L/\partial \mathbf{x} = W^\top\, \partial L/\partial \mathbf{o}$: the
backward pass applies the **transpose** of the forward operator. The same transpose appears in
adjoint methods, if you have met them.

Finally, a ReLU passes the gradient where its input was positive and blocks it where it was not:

$$\frac{\partial L}{\partial O(i,j)} = \frac{\partial L}{\partial A(i,j)} \times \begin{cases} 1 & O(i,j) > 0 \\ 0 & \text{otherwise} \end{cases}$$

### 4.3 The smallest example that has everything

Take the 6x6 piece of a real microstructure from Part 2, the Sobel kernel with bias $b = -0.5$, a
ReLU, and average the 16 outputs to get one prediction $\hat{y}$. The target is $y = 1$ and the loss
is $L = (\hat{y} - y)^2$. The cell computes every step by hand and then asks PyTorch's automatic
differentiation for the same numbers.

In [ ]:
# --- Forward and backward by hand, then checked ----------------------------------
I_bp = X6.copy()
K_bp = K_SOBEL.copy()
b_bp, y_bp = -0.5, 1.0

# forward
O_bp = conv2d_valid(I_bp, K_bp) + b_bp
A_bp = np.maximum(O_bp, 0)
yhat = A_bp.mean()
L_bp = (yhat - y_bp) ** 2

# backward
dL_dyhat = 2 * (yhat - y_bp)
dA = np.full((4, 4), dL_dyhat / 16)               # the mean shares it equally
dO = dA * (O_bp > 0)                               # ReLU gate
dK = np.zeros((3, 3)); dI = np.zeros((6, 6))
for i in range(4):
    for j in range(4):
        dK += dO[i, j] * I_bp[i:i+3, j:j+3]          # rule 1
        dI[i:i+3, j:j+3] += dO[i, j] * K_bp          # rule 3
db = dO.sum()                                        # rule 2

np.set_printoptions(precision=4, suppress=True)
print("input I\n", I_bp)
print("O = I * K + b\n", O_bp)
print(f"prediction y_hat = mean(ReLU(O)) = {yhat:.4f},  target {y_bp},  loss {L_bp:.4f}")
print(f"\ndL/dy_hat = 2 (y_hat - y) = {dL_dyhat:.4f}")
print("dL/dO (zero where the ReLU was off)\n", dO)
print("dL/dK\n", dK)
print(f"dL/db = {db:.4f}")

# the same with automatic differentiation
It = torch.tensor(I_bp, requires_grad=True)
Kt = torch.tensor(K_bp, requires_grad=True)
bt = torch.tensor(b_bp, requires_grad=True)
Lt = (torch.relu(F.conv2d(It[None, None], Kt[None, None]) + bt).mean() - y_bp) ** 2
Lt.backward()
print("\nlargest difference from autograd:")
print(f"  dL/dK {np.abs(dK - Kt.grad.numpy()).max():.1e}   dL/db {abs(db - bt.grad.item()):.1e}"
      f"   dL/dI {np.abs(dI - It.grad.numpy()).max():.1e}")

# the transpose check: build W for the Sobel kernel as in Part 2
W_sob = np.zeros((16, 36))
for p, (i, j) in enumerate(POS6):
    w = np.zeros((6, 6)); w[i:i+3, j:j+3] = K_bp
    W_sob[p] = w.ravel()
print(f"  W transpose times dL/dO equals dL/dI: largest difference "
      f"{np.abs(W_sob.T @ dO.ravel() - dI.ravel()).max():.1e}")


### Animation 9: the forward and backward pass, number by number

**Top row, forward:** kernel and bias, input, the convolution output $O$, and $A$ after the ReLU.
**Bottom row, backward:** the gradient of the loss with respect to each of those. **Right:** the
scalar steps.

The forward pass runs first (blue title), then the backward pass (red title). The line under the
title says what is happening in each frame. The frames to study most closely are the kernel
gradient ones: the window slides over the input a second time, and each patch is added to $\partial L/\partial K$ with
the weight shown in the orange cell of $\partial L/\partial O$. Where that weight is zero, the patch
adds nothing.

In [ ]:
# --- Animation 9: forward and backward through one convolution -------------------
fig = plt.figure(figsize=(14.5, 7.4), dpi=ANIM_DPI)
gsp = fig.add_gridspec(2, 5, width_ratios=[3, 6, 4.2, 4.2, 4.4], hspace=0.32, wspace=0.3,
                       left=0.02, right=0.99, top=0.84, bottom=0.03)
gK9 = number_grid(fig.add_subplot(gsp[0, 0]), (3, 3), vmax=4, title=f"kernel $K$, bias $b$ = {b_bp:g}")
gI9 = number_grid(fig.add_subplot(gsp[0, 1]), (6, 6), vmax=4, title="input $I$ (1 = fibre)")
gO9 = number_grid(fig.add_subplot(gsp[0, 2]), (4, 4), vmax=4, title=r"$O = I \star K + b$")
gA9 = number_grid(fig.add_subplot(gsp[0, 3]), (4, 4), vmax=4, title=r"$A = \mathrm{ReLU}(O)$")
gdK = number_grid(fig.add_subplot(gsp[1, 0]), (3, 3), cmap="PiYG", vmax=0.6, title=r"$\partial L/\partial K$")
gdI = number_grid(fig.add_subplot(gsp[1, 1]), (6, 6), cmap="PiYG", vmax=0.3, title=r"$\partial L/\partial I$", fontsize=8)
gdO = number_grid(fig.add_subplot(gsp[1, 2]), (4, 4), cmap="PiYG", vmax=0.15, title=r"$\partial L/\partial O$", fontsize=8)
gdA = number_grid(fig.add_subplot(gsp[1, 3]), (4, 4), cmap="PiYG", vmax=0.15, title=r"$\partial L/\partial A$", fontsize=8)
axS = fig.add_subplot(gsp[:, 4]); axS.axis("off")
txt_f = axS.text(0.02, 0.97, "", va="top", fontsize=10, family="monospace", color=C_DATA)
txt_b = axS.text(0.02, 0.45, "", va="top", fontsize=10, family="monospace", color=C_BAD)
head9 = fig.text(0.02, 0.955, "", fontsize=13, weight="bold")
note9 = fig.text(0.02, 0.91, "", fontsize=11)
fill_grid(gK9, K_bp); fill_grid(gI9, I_bp)
E3, E4, E6 = (np.full(s, np.nan) for s in [(3, 3), (4, 4), (6, 6)])
marks = []

FR9  = [("F", "start", 0)] + [("F", "conv", r) for r in range(4)] + [("F", "relu", 0)] + [("F", "mean", 0)] * 2
FR9 += [("B", "dyhat", 0), ("B", "dA", 0), ("B", "gate", 0), ("B", "gate", 0)]
FR9 += [("B", "dK", n) for n in range(16)] + [("B", "db", 0)] * 2
FR9 += [("B", "dI", r) for r in range(4)] + [("B", "done", 0)] * 4
F4, G4 = "{:+.1f}", "{:+.3f}"

def frame_bp(f):
    ph, kind, arg = FR9[f]
    while marks:
        marks.pop().remove()
    for g, e in [(gdK, E3), (gdI, E6), (gdO, E4), (gdA, E4)]:
        fill_grid(g, e)
    if kind == "start":
        fill_grid(gO9, E4)
    elif kind == "conv":
        o = E4.copy(); o[:arg + 1] = O_bp[:arg + 1]; fill_grid(gO9, o, F4)
    else:
        fill_grid(gO9, O_bp, F4)
    fill_grid(gA9, A_bp if (ph == "B" or kind in ("relu", "mean")) else E4, F4)
    fwd_lines, bwd_lines = [], []
    if ph == "B" or kind == "mean":
        fwd_lines = ["forward", "", "y_hat = mean(A)", f"      = {yhat:.4f}", f"y     = {y_bp:g}",
                     "L     = (y_hat - y)^2", f"      = {L_bp:.4f}"]
    if ph == "F":
        head9.set_text("FORWARD PASS"); head9.set_color(C_DATA)
        if kind == "start":
            note9.set_text("A 6x6 piece of a real microstructure. Ten numbers to learn: nine weights and one bias.")
        if kind == "conv":
            note9.set_text(f"Convolution, output row {arg}: the window visits four positions along this row.")
            outline(gI9["ax"], arg, 0, 3, 6, store=marks); outline(gO9["ax"], arg, 0, 1, 4, store=marks)
        if kind == "relu":
            note9.set_text("ReLU: negative entries become zero. Black boxes mark the units that are off.")
            for r, c in zip(*np.nonzero(O_bp <= 0)):
                outline(gO9["ax"], r, c, color="k", lw=2, store=marks)
                outline(gA9["ax"], r, c, color="k", lw=2, store=marks)
        if kind == "mean":
            note9.set_text("Average the 16 values to get the prediction, and compare it with the target.")
    else:
        head9.set_text("BACKWARD PASS"); head9.set_color(C_BAD)
        bwd_lines = ["backward", "", "dL/dy_hat = 2(y_hat - y)", f"          = {dL_dyhat:+.4f}"]
        if kind == "dyhat":
            note9.set_text("Start at the loss: how fast does L change when the prediction changes?")
        else:
            fill_grid(gdA, dA, G4)
        if kind == "dA":
            note9.set_text("The mean treats all 16 cells alike, so each one gets dL/dy_hat divided by 16.")
        if kind in ("gate", "dK", "db", "dI", "done"):
            fill_grid(gdO, dO, G4)
        if kind == "gate":
            note9.set_text("ReLU backward: the gradient passes where O > 0 and is blocked where the unit was off.")
            for r, c in zip(*np.nonzero(O_bp <= 0)):
                outline(gdO["ax"], r, c, color="k", lw=2, store=marks)
                outline(gO9["ax"], r, c, color="k", lw=2, store=marks)
        if kind == "dK":
            i, j = divmod(arg, 4)
            acc = np.zeros((3, 3))
            for t in range(arg + 1):
                ii, jj = divmod(t, 4)
                acc += dO[ii, jj] * I_bp[ii:ii+3, jj:jj+3]
            fill_grid(gdK, acc, G4)
            outline(gI9["ax"], i, j, 3, 3, store=marks); outline(gdO["ax"], i, j, store=marks)
            if dO[i, j] == 0:
                note9.set_text(f"dL/dK, position ({i},{j}): this unit was off, its gradient is 0, so the patch adds nothing.")
            else:
                note9.set_text(f"dL/dK, position ({i},{j}): add the patch under the window, times {dO[i, j]:+.4f}.")
        if kind in ("db", "dI", "done"):
            fill_grid(gdK, dK, G4)
            bwd_lines += ["", "dL/db = sum of dL/dO", f"      = {db:+.4f}"]
        if kind == "db":
            note9.set_text("The bias was added at all 16 positions, so its gradient is the sum of dL/dO. The layer can now update.")
        if kind in ("dI", "done"):
            rr = arg if kind == "dI" else 3
            acc = np.zeros((6, 6))
            for ii in range(rr + 1):
                for jj in range(4):
                    acc[ii:ii+3, jj:jj+3] += dO[ii, jj] * K_bp
            fill_grid(gdI, acc, "{:+.2f}")
            if kind == "dI":
                note9.set_text(f"dL/dI, row {rr}: each position sends K times its gradient back onto its own patch. Overlaps add.")
                outline(gdI["ax"], rr, 0, 3, 6, store=marks); outline(gdO["ax"], rr, 0, 1, 4, store=marks)
        if kind == "done":
            note9.set_text("dL/dK and dL/db update this layer. dL/dI is passed down to the layer before.")
            bwd_lines += ["", "all of this matches", "PyTorch autograd,", "see the cell above"]
    txt_f.set_text("\n".join(fwd_lines)); txt_b.set_text("\n".join(bwd_lines))
    return []

anim = animation.FuncAnimation(fig, frame_bp, frames=len(FR9), interval=1000)
play(anim, fig)


**What the animation shows.** The forward pass is the arithmetic of Part 1 with a ReLU and an
average added. The backward pass runs the same diagram from right to left.

Points to note:

- The backward pass starts from one number, $\partial L/\partial \hat{y}$, and every other gradient is
  obtained from it by multiplication.
- Units that the ReLU switched off pass nothing back, so their patches add nothing to
  $\partial L/\partial K$.
- $\partial L/\partial K$ is built by sliding the same window over the same input again, so the forward
  pass and the kernel gradient are the same operation with different weights.

A full CNN does the same with many more kernels, channels and layers, and PyTorch keeps track of all
of it.

### 4.4 Watching a kernel learn

Next, the gradient is used for training. 200 random 16x16 pieces of microstructure are passed through
a **hidden** kernel, the five-point Laplacian stencil, to make 200 target maps. A single $3 \times 3$
kernel with a bias starts from random numbers and is trained by plain gradient descent to reproduce
the targets. It is never shown the hidden kernel, only the input and output pictures. The cells below
check whether it ends up with the stencil.

In [ ]:
# --- Train one kernel to reproduce a hidden stencil ------------------------------
rng = np.random.default_rng(0)
n_crop = 200
ci = rng.integers(0, len(X_IMG), n_crop)
cr = rng.integers(0, 48, n_crop); cc = rng.integers(0, 48, n_crop)
CROPS = torch.tensor(np.stack([X_IMG[i, a:a+16, b:b+16] for i, a, b in zip(ci, cr, cc)])[:, None])
K_HIDDEN = torch.tensor(K_LAP)[None, None]
TARGETS = F.conv2d(CROPS, K_HIDDEN)                      # 200 target maps, 14x14 each

def learn_kernel(lr=0.25, steps=1000, seed=SEED, keep=()):
    torch.manual_seed(seed)
    K = (0.5 * torch.randn(1, 1, 3, 3)).requires_grad_()
    b = torch.zeros(1, requires_grad=True)
    losses, snaps = [], {}
    for s in range(steps + 1):
        pred = F.conv2d(CROPS, K, b)
        L = ((pred - TARGETS) ** 2).mean()
        losses.append(L.item())
        if s in keep:
            snaps[s] = (K.detach()[0, 0].numpy().copy(), b.item(), pred[0, 0].detach().numpy().copy())
        if not np.isfinite(losses[-1]) or s == steps:
            break
        gK, gb = torch.autograd.grad(L, [K, b])
        with torch.no_grad():
            K -= lr * gK
            b -= lr * gb
    return np.array(losses), snaps, K.detach()[0, 0].numpy()

KEEP = [0, 1, 2, 3, 5, 8, 12, 18, 25, 35, 50, 70, 100, 140, 200, 280, 400, 550, 750, 1000]
t0 = time.time()
LOSS_K, SNAPS, K_LEARNED = learn_kernel(keep=KEEP)
print(f"1000 steps of gradient descent in {time.time() - t0:.2f} s")
print(f"loss: start {LOSS_K[0]:.3f}, after 100 steps {LOSS_K[100]:.4f}, after 1000 steps {LOSS_K[-1]:.2e}")
print("learned kernel:\n", np.round(K_LEARNED, 3))
print(f"learned bias: {SNAPS[1000][1]:+.4f}")


### Animation 10: the kernel converging

**Left:** the hidden stencil. **Second:** the kernel being trained, with its current numbers.
**Third and fourth:** the target map for one of the 200 pieces and the kernel's current output for
it. **Right:** the loss against the step number, on logarithmic axes.

In [ ]:
# --- Animation 10: a kernel learning the Laplacian --------------------------------
fig = plt.figure(figsize=(15, 4.6), dpi=ANIM_DPI)
gsp = fig.add_gridspec(1, 5, width_ratios=[1, 1, 1, 1, 1.6], wspace=0.45, top=0.8, bottom=0.14, left=0.02, right=0.98)
fig.subplots_adjust(wspace=0.45)
gH = number_grid(fig.add_subplot(gsp[0]), (3, 3), vmax=4, title="hidden stencil", fontsize=12)
fill_grid(gH, K_LAP)
gL10 = number_grid(fig.add_subplot(gsp[1]), (3, 3), vmax=4, title="", fontsize=11)
axT = fig.add_subplot(gsp[2]); axP = fig.add_subplot(gsp[3])
tmap = TARGETS[0, 0].numpy(); vmt = np.abs(tmap).max()
axT.imshow(tmap, cmap="coolwarm", vmin=-vmt, vmax=vmt, interpolation="nearest")
axT.set_title("target map", fontsize=10); bare(axT)
imP = axP.imshow(np.zeros_like(tmap), cmap="coolwarm", vmin=-vmt, vmax=vmt, interpolation="nearest")
axP.set_title("output of the kernel being trained", fontsize=10); bare(axP)
axLc = fig.add_subplot(gsp[4])
steps_all = np.arange(len(LOSS_K))
axLc.loglog(steps_all[1:], LOSS_K[1:], color="0.8", lw=1)
dot, = axLc.loglog([], [], "o", color=C_FIT, ms=8)
trail, = axLc.loglog([], [], color=C_FIT, lw=2)
axLc.set_xlabel("gradient descent step"); axLc.set_ylabel("loss (mean squared error)")
axLc.set_title("loss", fontsize=10)
head10 = fig.text(0.02, 0.92, "", fontsize=13, weight="bold")

FR10 = KEEP + [KEEP[-1]] * 4

def frame_learn(f):
    s = FR10[f]
    K_s, b_s, p_s = SNAPS[s]
    fill_grid(gL10, K_s, "{:+.2f}")
    gL10["ax"].set_title(f"trained kernel, b = {b_s:+.2f}", fontsize=10)
    imP.set_data(p_s)
    lo = max(s, 1)
    trail.set_data(steps_all[1:lo + 1], LOSS_K[1:lo + 1])
    dot.set_data([lo], [LOSS_K[s]])
    head10.set_text(f"step {s}:  loss = {LOSS_K[s]:.2e}")
    return []

anim = animation.FuncAnimation(fig, frame_learn, frames=len(FR10), interval=600)
play(anim, fig)


**What the animation shows.** The trained kernel starts as nine random numbers and an output
that looks nothing like the target. Within a hundred or so steps the centre weight has gone
strongly negative and the four direct neighbours positive: the shape of the stencil appears before
the numbers are right. After that the remaining error shrinks steadily. By the end, the printed
kernel is the five-point Laplacian, and the loss has fallen by many orders of magnitude.

No Taylor series was used: the stencil came from 200 pairs of pictures. In a real CNN there is no
hidden kernel to find. The kernels move towards whatever helps predict the target, and Notebook 3b
looks at what they become.

### Try it: the learning rate

The same experiment with a learning rate of your choice. For this problem there is a largest stable
learning rate, and going past it makes the weights blow up, as the diverging learning rate did in
Notebook 1. Try 0.05, 0.25
and 0.3.

In [ ]:
# --- Interactive: learning rate for the kernel ----------------------------------
def lr_explorer(lr=0.25, steps=600):
    losses, _, K_end = learn_kernel(lr=lr, steps=steps)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.8, 1]})
    ok = np.isfinite(losses)
    a1.semilogy(np.arange(len(losses))[ok], losses[ok], color=C_DATA, lw=2)
    a1.set_xlabel("step"); a1.set_ylabel("loss")
    diverged = (not np.isfinite(losses[-1])) or losses[-1] > losses[0]
    a1.set_title(f"learning rate {lr}: " + ("diverged" if diverged else f"final loss {losses[-1]:.2e}"),
                 fontsize=10, color=C_BAD if diverged else "k")
    if diverged:
        a2.text(0.5, 0.5, "weights blew up", ha="center", va="center", fontsize=13, color=C_BAD)
        bare(a2)
    else:
        g = number_grid(a2, (3, 3), vmax=4, title="kernel at the end", fontsize=12)
        fill_grid(g, K_end, "{:+.2f}")
    plt.tight_layout(); plt.show()

interact(lr_explorer,
         lr=Dropdown(options=[0.01, 0.05, 0.1, 0.2, 0.25, 0.28, 0.3, 0.5], value=0.25),
         steps=IntSlider(value=600, min=50, max=1500, step=50, continuous_update=False));


### Summary of Part 4

- The gradient of the loss with respect to the kernel is a convolution of the input with the map of
  output gradients.
- The gradient passed to the layer below applies the transpose of the forward operator.
- A ReLU blocks the gradient at units that were off, and max pooling sends it to the winner only.
- With these rules, plain gradient descent recovered the Laplacian stencil from examples alone.

---

# Part 5 - Why this helps: two small experiments

Parts 1 to 4 built the layer. This part checks, with measurements, that the two design choices
(local connections and shared weights) actually change what a network can do.

### 5.1 A pattern in a place the network has never seen

This is a small classification task. Each image is 24x24 pixels of faint noise with one short bar in it. The
bar is either horizontal (label 1) or vertical (label 0). The network must say which.

In the **training** images the bar is always in the **left** half. The network is then
tested twice, on new images with the bar in the left half, and on images with the bar in the
**right** half, where it has never seen one.

Two models are trained on the same data:

- a dense network, as in Notebook 2: flatten, 64 neurons, one output;
- a small CNN: two $3 \times 3$ convolution layers with 8 kernels each, then **global max pooling**,
  which keeps the largest value of each channel over the whole image, then one output neuron.

Global max pooling answers "did this detector fire anywhere?", which does not depend on where.

Both output a number that the sigmoid of Notebook 1 turns into a probability $p$ that the bar is
horizontal. Both are trained with the **binary cross-entropy** loss, the usual loss for a yes or no
answer:

$$L = -\big[\,y \ln p + (1 - y)\ln(1 - p)\,\big]$$

where $y$ is 1 for a horizontal bar and 0 for a vertical one. The loss is small when $p$ is close to
$y$ and large when the network is confident and wrong.

In [ ]:
# --- Train a dense network and a CNN on bars in the left half -------------------
N_BAR = 24
np.random.seed(SEED); torch.manual_seed(SEED)

def bar_images(n, half):
    X = np.zeros((n, 1, N_BAR, N_BAR), np.float32); y = np.zeros(n, np.float32)
    for t in range(n):
        lab = np.random.randint(2); y[t] = lab
        r = np.random.randint(2, N_BAR - 2)
        if half == "left":
            c = np.random.randint(2, N_BAR // 2 - 2)
        else:
            c = np.random.randint(N_BAR // 2 + 2, N_BAR - 2)
        if lab:
            X[t, 0, r, c-2:c+3] = 1          # horizontal bar, 5 pixels long
        else:
            X[t, 0, r-2:r+3, c] = 1          # vertical bar
    X += 0.15 * np.random.rand(*X.shape).astype(np.float32)
    return torch.tensor(X), torch.tensor(y)

Xb_tr, yb_tr = bar_images(2000, "left")
Xb_L,  yb_L  = bar_images(1000, "left")
Xb_R,  yb_R  = bar_images(1000, "right")

dense_net = nn.Sequential(nn.Flatten(), nn.Linear(N_BAR * N_BAR, 64), nn.ReLU(), nn.Linear(64, 1))
conv_net  = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
                          nn.Conv2d(8, 8, 3, padding=1), nn.ReLU(),
                          nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(8, 1))

BAR_RES = {}
for name, m in [("dense network", dense_net), ("CNN", conv_net)]:
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    t0 = time.time()
    for ep in range(30):
        perm = torch.randperm(len(Xb_tr))
        for i in range(0, len(Xb_tr), 64):
            j = perm[i:i + 64]
            opt.zero_grad()
            loss = F.binary_cross_entropy_with_logits(m(Xb_tr[j]).squeeze(1), yb_tr[j])
            loss.backward(); opt.step()
    with torch.no_grad():
        acc_L = ((m(Xb_L).squeeze(1) > 0).float() == yb_L).float().mean().item()
        acc_R = ((m(Xb_R).squeeze(1) > 0).float() == yb_R).float().mean().item()
    BAR_RES[name] = (sum(p.numel() for p in m.parameters()), acc_L, acc_R, time.time() - t0)

print(f"{'model':15s} {'weights':>8s} {'accuracy, left half':>21s} {'accuracy, right half':>22s} {'time':>7s}")
for name, (npar, aL, aR, el) in BAR_RES.items():
    print(f"{name:15s} {npar:>8,d} {aL:>21.3f} {aR:>22.3f} {el:>6.1f}s")
print("an accuracy of 0.5 is what guessing gives")


### Animation 11: sliding one bar across the image

Take one horizontal bar and one vertical bar at the same row, and move them across the image one
column at a time. At every position, ask both trained networks for the probability that the bar is
horizontal.

**Left:** the current test image. The shaded band marks the columns where training bars were
placed. **Right:** the answers. Solid lines are for the horizontal bar (the right answer is 1),
dashed lines for the vertical bar (the right answer is 0).

In [ ]:
# --- Animation 11: the same bar at every column ----------------------------------
rng_probe = np.random.default_rng(7)
noise = 0.15 * rng_probe.random((N_BAR, N_BAR)).astype(np.float32)
COLS = list(range(2, N_BAR - 2))
ROW = 12

def probe(c, horizontal):
    x = noise.copy()
    if horizontal:
        x[ROW, c-2:c+3] += 1
    else:
        x[ROW-2:ROW+3, c] += 1
    return x

PROBS = {}
with torch.no_grad():
    for name, m in [("dense network", dense_net), ("CNN", conv_net)]:
        for hz in (True, False):
            xs_ = torch.tensor(np.stack([probe(c, hz) for c in COLS]))[:, None]
            PROBS[(name, hz)] = torch.sigmoid(m(xs_)).squeeze(1).numpy()

outside = np.array(COLS) > N_BAR // 2 + 1        # bar centres the training data never used
print("average P(horizontal) for bars centred outside the training columns:")
for name in ("dense network", "CNN"):
    print(f"  {name:14s} horizontal bar {PROBS[(name, True)][outside].mean():.3f}   "
          f"vertical bar {PROBS[(name, False)][outside].mean():.3f}")
print("right answers: 1 for the horizontal bar, 0 for the vertical bar")

fig, (axI, axC) = plt.subplots(1, 2, figsize=(12, 4.6), dpi=ANIM_DPI,
                               gridspec_kw={"width_ratios": [1, 1.6]})
imI = axI.imshow(probe(COLS[0], True), cmap="gray", vmin=0, vmax=1.15, interpolation="nearest")
axI.axvspan(-0.5, N_BAR // 2 - 0.5, color=C_ALT, alpha=0.15)
axI.text(1, 22.5, "training bars lived here", color=C_ALT, fontsize=9)
bare(axI)
axC.axvspan(1.5, N_BAR // 2 - 2.5, color=C_ALT, alpha=0.15)
lines = {}
for name, col in [("dense network", C_BAD), ("CNN", C_DATA)]:
    for hz, ls in [(True, "-"), (False, "--")]:
        lines[(name, hz)], = axC.plot([], [], ls, color=col, lw=5 if name == "dense network" else 2,
                                      alpha=0.75 if name == "dense network" else 1,
                                      label=f"{name}, {'horizontal' if hz else 'vertical'} bar")
axC.set_xlim(COLS[0] - 0.5, COLS[-1] + 0.5); axC.set_ylim(-0.05, 1.05)
axC.set_xlabel("column of the bar centre"); axC.set_ylabel("P(horizontal)")
axC.legend(fontsize=8, loc="center right")
seq = [(k, True) for k in range(len(COLS))] + [(k, False) for k in range(len(COLS))]

def frame_bar(f):
    k, hz = seq[f]
    imI.set_data(probe(COLS[k], hz))
    axI.set_title(f"{'horizontal' if hz else 'vertical'} bar at column {COLS[k]}", fontsize=10)
    for (name, h_), ln in lines.items():
        if h_ == hz:
            ln.set_data(COLS[:k + 1], PROBS[(name, h_)][:k + 1])
        elif h_ and not hz:
            ln.set_data(COLS, PROBS[(name, h_)])
        else:
            ln.set_data([], [])
    return []

anim = animation.FuncAnimation(fig, frame_bar, frames=len(seq), interval=220)
play(anim, fig)


**What the table and the animation show.** Both models are near perfect on bars in the left half,
where they were trained. On the right half the dense network falls to about the accuracy of a coin
toss, while the CNN keeps its accuracy. The printed table gives the numbers, and the CNN gets there
with far fewer weights.

In the animation, the CNN's lines stay flat across the whole image: the answer does not depend on
where the bar is. The dense network (thick lines) gets both bars right inside the shaded band.
Outside it, compare the printed averages with the right answers: at least one of the two bars is
now misread. The dense network learned "pixels in these places", because every
pixel has its own weights. The CNN learned "a horizontal run of bright pixels", with one kernel
used everywhere. This comes from weight sharing, and it is one of the main reasons CNNs are used on
images.

### 5.2 The shuffled-pixel test, explained

Notebook 2 applied one fixed random reordering, a **permutation**, to the pixels of every image,
training and test alike. The dense network scored the same on the shuffled images as on the real
ones. The first two points below explain that result, and the third predicts what a CNN will do.

1. **No information is lost.** The same permutation is used for every image, so it can be undone.
   The fibre fraction is unchanged too.
2. **A dense layer cannot tell the difference.** If $P$ is the permutation, then
   $(W P^\top)(P\mathbf{x}) = W\mathbf{x}$: reordering the weights in the same way undoes the
   shuffle exactly. Whatever the network could learn from real images, it can learn from shuffled
   ones.
3. **A convolution can tell the difference.** It assumes that the pixels inside a $3 \times 3$
   window belong together. After the shuffle, the nine pixels in a window come from nine unrelated
   places in the cell. The assumption that made the CNN work in 5.1 is now false.

The animation applies the Sobel-$x$ kernel of Part 1, with a ReLU, to a real microstructure and to
its shuffled version.

In [ ]:
# --- Animation 12: the same kernel on a real and a shuffled image ------------------
PERM = np.random.default_rng(1).permutation(64 * 64)     # the same permutation as Notebooks 2 and 3b
img_real = X_IMG[IMG_DEMO]
img_shuf = img_real.ravel()[PERM].reshape(64, 64)
FM_REAL = np.maximum(conv2d_valid(img_real, K_SOBEL), 0)
FM_SHUF = np.maximum(conv2d_valid(img_shuf, K_SOBEL), 0)
print(f"fibre fraction: real {img_real.mean():.4f}, shuffled {img_shuf.mean():.4f}")
print(f"units active after ReLU: real {100 * (FM_REAL > 0).mean():.1f}%, "
      f"shuffled {100 * (FM_SHUF > 0).mean():.1f}%")

fig, axes = plt.subplots(2, 2, figsize=(10, 9.4), dpi=ANIM_DPI)
vmx = max(FM_REAL.max(), FM_SHUF.max())
handles = []
for c, (img_, fm_, title) in enumerate([(img_real, FM_REAL, "real microstructure"),
                                        (img_shuf, FM_SHUF, "same pixels, shuffled")]):
    axes[0, c].imshow(img_, cmap="gray", interpolation="nearest")
    axes[0, c].set_title(f"{title}\nfibre fraction {img_.mean():.3f}", fontsize=10)
    band = Rectangle((-0.5, -0.5), 64, 3, fill=False, ec=C_FIT, lw=2.5)
    axes[0, c].add_patch(band)
    cm_ = plt.get_cmap("magma").copy(); cm_.set_bad("0.85")
    im_ = axes[1, c].imshow(np.full(fm_.shape, np.nan), cmap=cm_, vmin=0, vmax=vmx,
                            interpolation="nearest")
    axes[1, c].set_title("Sobel x, then ReLU", fontsize=10)
    for a in axes[:, c]:
        bare(a)
    handles.append((band, im_, fm_))

ROWS_PER = 4
N12 = int(np.ceil(FM_REAL.shape[0] / ROWS_PER))

def frame_shuf(f):
    upto = min((f + 1) * ROWS_PER, FM_REAL.shape[0])
    for band, im_, fm_ in handles:
        part = np.full(fm_.shape, np.nan)
        part[:upto] = fm_[:upto]
        im_.set_data(part)
        band.set_y(upto - ROWS_PER - 0.5)
        band.set_height(ROWS_PER + 2)
    return []

anim = animation.FuncAnimation(fig, frame_shuf, frames=N12 + 3, interval=250)
play(anim, fig)


**What the animation shows.** On the real image the kernel responds along the fibre boundaries
and is silent elsewhere, so the bottom-left map traces the left edges of the fibres. On the shuffled
image it responds all over the cell, in short scattered streaks, because most windows now contain a
random mix of fibre and matrix. The printed numbers confirm that the fibre fraction is identical,
while the share of active units is very different.

The last figure shows where the nine pixels of one window in the shuffled image originally were.

In [ ]:
# --- Where do the pixels of one shuffled window come from? --------------------------
wi, wj = 30, 30                                        # a 3x3 window in the shuffled image
flat_pos = [r * 64 + c for r in range(wi, wi + 3) for c in range(wj, wj + 3)]
origin = [divmod(int(PERM[p]), 64) for p in flat_pos]  # the shuffled pixel p came from PERM[p]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.9))
axes[0].imshow(img_shuf, cmap="gray", interpolation="nearest")
axes[0].add_patch(Rectangle((wj - 0.5, wi - 0.5), 3, 3, fill=False, ec=C_FIT, lw=3))
axes[0].set_title("shuffled image: one 3x3 window", fontsize=10)
axes[1].imshow(img_real, cmap="gray", interpolation="nearest")
for (r, c) in origin:
    axes[1].plot(c, r, "o", ms=9, mfc="none", mec=C_FIT, mew=2.5)
axes[1].add_patch(Rectangle((wj - 0.5, wi - 0.5), 3, 3, fill=False, ec=C_ALT, lw=2, ls="--"))
axes[1].set_title("real image: where those nine pixels came from", fontsize=10)
for a in axes:
    bare(a)
plt.tight_layout(); plt.show()


**What the figure shows.** Left: nine pixels that sit side by side in the shuffled image. Right:
the orange circles are the places in the real cell those pixels came from, scattered across the
whole cell. The green dashed box marks the same window position in the real image, for comparison.
Any kernel reading the left window is combining pixels that have nothing to do with each other.

This gives a prediction for Notebook 3b: a CNN trained on shuffled images should do badly whenever the
answer depends on how the fibres are arranged. On a target that depends only on how much fibre
there is, shuffling should not matter, because the fibre fraction survives. Notebook 3b runs the
test.

---

# What to take away

1. A convolution slides a small kernel over the input and writes one weighted sum per position.
   Finite-difference stencils are convolution kernels.
2. A convolution layer is a dense layer with local connections and shared weights. Its number of
   weights depends on the kernel size, not on the image size.
3. Padding, stride, channels and pooling are the other building blocks. For a periodic unit cell, use
   circular padding: it is the periodic boundary condition.
4. Stacking layers grows the receptive field, and pooling makes it grow fast. A unit can only
   use what lies inside its field.
5. The kernel gradient is a convolution of the input with the output gradients, and the input
   gradient applies the transpose. Gradient descent with these rules recovered the Laplacian
   stencil from examples.
6. Weight sharing is why a CNN recognises a pattern in a place it has never seen it. The same
   assumption, that neighbouring pixels belong together, is what shuffling breaks.

# Exercises

### 1. Output sizes by hand
Using the formula of Part 3, work out the output side for a 64x64 input with
(a) $k=5, p=2, s=1$, (b) $k=3, p=1, s=2$, (c) $k=7, p=0, s=3$. Check each with `nn.Conv2d`.

### 2. Count the weights of the Notebook 3b network
Use $P_{\text{conv}} = k^2 c_{in} c_{out} + c_{out}$ and $P_{\text{dense}} = n_{in} n_{out} + n_{out}$
to count the weights of the network in Part 3.6, layer by layer, by hand. Compare your total with
`N_TOTAL`. Which layer holds most of them?

### 3. A detector for one kind of boundary
In the 1D example of Part 1, find a kernel and a bias such that, after a ReLU, the output is
positive only where the row **enters** a fibre, and zero everywhere else. Check it with
`conv1d_valid`.

### 4. Learn a different stencil
In Part 4.4, replace `K_HIDDEN` by the Sobel-$x$ kernel and rerun. Is it recovered? Then replace it
by the $5 \times 5$ kernel `torch.ones(1, 1, 5, 5) / 25` while still training a $3 \times 3$ kernel.
Adjust the code so that the target and the output have the same size. What happens to the loss,
and why can it not reach zero?

### 5. Receptive field without pooling
Remove the three pooling layers from `LAYERS` in Part 3.5. Use the formula to find the receptive
field after the three convolutions. How many $3 \times 3$ convolution layers without pooling would
be needed to reach the field of the original network? Check one case with `measured_patch`.

### 6. Is a CNN always the right tool?
Suppose the task is to predict the fibre fraction of a microstructure from its image. Train a small
CNN on it, using the network of 5.1 with a regression output and the mean squared error. Then
compute the answer with one line, `X_IMG.mean(axis=(1, 2))`. Which is more accurate, and which would
you use? What does this tell you about when learning features from images is worth the effort?

---

### Aside: convolution or cross-correlation?

Strictly, the operation in this notebook is a **cross-correlation**. The textbook convolution flips
the kernel first:

$$(I * K)(i,j) = \sum_m \sum_n I(i-m,\; j-n)\, K(m,n)$$

Every deep learning library computes the unflipped version and calls it convolution. Since the
kernel is learned, the network learns the flipped weights if it needs them, and nothing
changes. The distinction only matters when comparing with a signal-processing result that assumes
the flip.